# Base B — Calidad de las ofertas
# Base B — EDA 01: inventario y estructura de snapshots

**Proyecto Final — Observatorio de Financiamiento y Endeudamiento en Argentina**

Objetivo de este notebook:

1. inventariar todos los snapshots disponibles;
2. verificar artefactos y completitud física de cada corrida;
3. documentar la estructura `raw → flattened → processed`;
4. inspeccionar schemas y tipos por familia;
5. distinguir campos provenientes del BCRA de campos agregados por el pipeline;
6. medir presencia y nulabilidad observada de campos;
7. revisar los artefactos de auditoría global (`manifest`, `quality_flags`, `key_collisions`, `ratio_normalization_audit`);
8. dejar contestadas las decisiones estructurales pendientes antes de iniciar el EDA temporal.

**Fuera de alcance:** altas/bajas/modificaciones entre snapshots, imputación, limpieza, cruces con Base A y transformaciones analíticas.


In [1]:
from pathlib import Path
import json, re
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# El notebook trabaja directamente sobre la carpeta data_b del repositorio.
# La ruta se resuelve desde el directorio del notebook.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_b").is_dir()), Path.cwd())
DATA_B = ROOT / "data_b"
if not DATA_B.is_dir():
    raise FileNotFoundError(f"No se encontró data_b en {ROOT}.")

# Por defecto el EDA incorpora TODOS los snapshots disponibles al momento de ejecutar.
# Sólo definir una fecha si se necesita reproducir un corte histórico específico.
EDA_CUTOFF_DATE = None  # ejemplo: "2026-09-16"

FAMILIES = ["hipotecarios", "personales", "prendarios", "tarjetas"]

print("data_b:", DATA_B.resolve())
print("Universo EDA:", f"hasta {EDA_CUTOFF_DATE}" if EDA_CUTOFF_DATE else "todos los snapshots disponibles")


data_b: C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cuatri\PROYECTO FINAL EN CIENCIA DE DATOS\scripts\data_b
Universo EDA: todos los snapshots disponibles



## 1. Inventario físico de snapshots

Se detectan directorios con patrón:

`data_b/snapshots/YYYY-MM-DD/<snapshot_id>/`

y se valida la presencia de:

- `raw/<familia>.json`
- `flattened/<familia>.csv`
- `processed/<familia>.csv`
- `processed/all_products.csv`
- `snapshot_metadata.json`

La completitud se determina **por artefactos físicos**, sin asumir todavía si una corrida debe entrar al panel analítico.


In [2]:
SNAPSHOT_ROOT = DATA_B / "snapshots"
snapshots = {
    (date_dir.name, snapshot_dir.name)
    for date_dir in SNAPSHOT_ROOT.iterdir()
    if date_dir.is_dir()
    for snapshot_dir in date_dir.iterdir()
    if snapshot_dir.is_dir() and re.fullmatch(r"\d{4}-\d{2}-\d{2}", date_dir.name)
}

rows = []
for date, sid in sorted(snapshots):
    base = SNAPSHOT_ROOT / date / sid
    row = {"snapshot_date_dir": date, "snapshot_id": sid}
    for fam in FAMILIES:
        row[f"raw_{fam}"] = (base / "raw" / f"{fam}.json").is_file()
        row[f"flat_{fam}"] = (base / "flattened" / f"{fam}.csv").is_file()
        row[f"proc_{fam}"] = (base / "processed" / f"{fam}.csv").is_file()
    row["all_products"] = (base / "processed" / "all_products.csv").is_file()
    row["metadata"] = (base / "snapshot_metadata.json").is_file()
    row["n_raw_families"] = sum(row[f"raw_{f}"] for f in FAMILIES)
    row["n_flat_families"] = sum(row[f"flat_{f}"] for f in FAMILIES)
    row["n_proc_families"] = sum(row[f"proc_{f}"] for f in FAMILIES)
    row["physically_complete"] = (
        row["n_raw_families"] == len(FAMILIES)
        and row["n_flat_families"] == len(FAMILIES)
        and row["n_proc_families"] == len(FAMILIES)
        and row["all_products"]
        and row["metadata"]
    )
    rows.append(row)

physical_inventory = pd.DataFrame(rows)

if physical_inventory.empty:
    raise RuntimeError("No se detectaron snapshots en data_b/snapshots.")

if EDA_CUTOFF_DATE is not None:
    physical_inventory = physical_inventory[
        physical_inventory["snapshot_date_dir"] <= EDA_CUTOFF_DATE
    ].copy()

if physical_inventory.empty:
    raise RuntimeError("El corte seleccionado no contiene snapshots.")

# Universo canónico del EDA. Todos los artefactos globales se filtran contra estos snapshot_id.
INCLUDED_SNAPSHOT_IDS = set(physical_inventory["snapshot_id"].astype(str))
INCLUDED_SNAPSHOT_DATES = set(physical_inventory["snapshot_date_dir"].astype(str))

def restrict_to_eda_universe(df):
    """
    Restringe un artefacto global exactamente al mismo universo de snapshots
    usado por el inventario físico. Prioriza snapshot_id; usa snapshot_date
    sólo como fallback.
    """
    out = df.copy()
    if "snapshot_id" in out.columns:
        return out[out["snapshot_id"].astype(str).isin(INCLUDED_SNAPSHOT_IDS)].copy()
    if "snapshot_date" in out.columns:
        return out[out["snapshot_date"].astype(str).isin(INCLUDED_SNAPSHOT_DATES)].copy()
    return out

display(
    physical_inventory[
        ["snapshot_date_dir","snapshot_id","n_raw_families","n_flat_families",
         "n_proc_families","all_products","metadata","physically_complete"]
    ]
)

print("\nUniverso EDA:", f"hasta {EDA_CUTOFF_DATE}" if EDA_CUTOFF_DATE else "todos los snapshots disponibles")
print("Directorios de snapshot:", len(physical_inventory))
print("Completos físicamente:", int(physical_inventory.physically_complete.sum()))
print("Parciales/incompletos:", int((~physical_inventory.physically_complete).sum()))


,snapshot_date_dir,snapshot_id,n_raw_families,n_flat_families,n_proc_families,all_products,metadata,physically_complete
0,2026-09-07,20260907_212935_569641,4,4,4,True,True,True
1,2026-09-08,20260908_224655_838862,4,4,4,True,True,True
2,2026-09-09,20260909_230424_341254,4,4,4,True,True,True
3,2026-09-10,20260910_221306_387342,4,4,4,True,True,True
4,2026-09-11,20260911_215454_602912,4,4,4,True,True,True
5,2026-09-12,20260912_192218_082470,4,4,4,True,True,True
6,2026-09-13,20260913_220415_571963,4,4,4,True,True,True
7,2026-09-14,20260914_231738_476390,4,4,4,True,True,True
8,2026-09-15,20260915_230313_952585,4,4,4,True,True,True
9,2026-09-16,20260916_235331_528166,4,4,4,True,True,True



Universo EDA: todos los snapshots disponibles
Directorios de snapshot: 12
Completos físicamente: 12
Parciales/incompletos: 0



## 2. Inventario registrado en `manifest.csv`

El `manifest` funciona como catálogo de corridas/familias completas. Se muestran:

- fecha;
- `snapshot_id`;
- familia;
- HTTP status;
- registros raw;
- duplicados exactos;
- registros processed;
- errores reportados.

Luego se pivotea a una fila por snapshot.


In [3]:
manifest_live = pd.read_csv(DATA_B / "manifest.csv")
manifest = restrict_to_eda_universe(manifest_live)

print("Columnas manifest:")
print(manifest.columns.tolist())

print("\nFilas manifest live:", len(manifest_live))
print("Filas manifest incluidas en EDA:", len(manifest))
display(manifest.head())

# Detección flexible de nombres
def pick(cols, *candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

cols = manifest.columns
date_col = pick(cols, "snapshot_date", "date")
sid_col = pick(cols, "snapshot_id")
fam_col = pick(cols, "producto", "family", "familia")
status_col = pick(cols, "http_status", "status")
raw_col = pick(cols, "raw_rows", "rows_raw", "raw_count")
proc_col = pick(cols, "processed_rows", "rows_processed", "processed_count")
dup_col = pick(cols, "duplicate_rows", "duplicates_removed", "exact_duplicates_removed", "exact_duplicates", "duplicates")
err_col = pick(cols, "error", "errors")

print("\nCampos detectados:")
print({
    "fecha": date_col, "snapshot_id": sid_col, "familia": fam_col,
    "http_status": status_col, "raw": raw_col, "processed": proc_col,
    "duplicados": dup_col, "error": err_col
})

display(manifest)


Columnas manifest:
['snapshot_date', 'snapshot_id', 'run_at', 'producto', 'http_status', 'raw_rows', 'exact_duplicates_removed', 'processed_rows', 'entidades', 'fecha_info_min', 'fecha_info_max', 'sha256', 'previous_sha256', 'changed_vs_previous', 'raw_path', 'flattened_path', 'processed_path', 'error']

Filas manifest live: 52
Filas manifest incluidas en EDA: 48


,snapshot_date,snapshot_id,run_at,producto,http_status,raw_rows,exact_duplicates_removed,processed_rows,entidades,fecha_info_min,fecha_info_max,sha256,previous_sha256,changed_vs_previous,raw_path,flattened_path,processed_path,error
0,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,200,321,51,270,30,2017-04-06T00:00:00,2026-09-07T00:00:00,d9d3028285c7f650ffe1bf3680ae0174e3a13c8ee76d04db7b44c707dd766963,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\hipotecarios.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\hipotecarios.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\hipotecarios.csv,NaN
1,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,personales,200,2380,98,2282,291,2020-05-04T00:00:00,2026-09-07T00:00:00,745d5b2593d73bb993076ba3b1af75088cea66d664e67c2f24d642e6982fed21,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\personales.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\personales.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\personales.csv,NaN
2,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,200,493,2,491,48,2017-04-06T00:00:00,2026-09-07T00:00:00,70f915558e14af2ed5b88ba9025b03659df697a4d4621261515f18b349edf7ae,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\prendarios.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\prendarios.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\prendarios.csv,NaN
3,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,tarjetas,200,541,0,541,137,2019-08-27T00:00:00,2026-09-07T00:00:00,875d3e24f710a07808ac63364fc9f6aa751c9e58a41cde3ebc4d910de514e639,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\tarjetas.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\tarjetas.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\tarjetas.csv,NaN
4,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,200,319,51,268,30,2017-04-06T00:00:00,2026-09-08T00:00:00,9bbce0150dd2565b74e1298827b1ab6ba06a498c7c4c859ca0c66f90bb2f0e29,d9d3028285c7f650ffe1bf3680ae0174e3a13c8ee76d04db7b44c707dd766963,True,data_b\snapshots\2026-09-08\20260908_224655_838862\raw\hipotecarios.json,data_b\snapshots\2026-09-08\20260908_224655_838862\flattened\hipotecarios.csv,data_b\snapshots\2026-09-08\20260908_224655_838862\processed\hipotecarios.csv,NaN



Campos detectados:
{'fecha': 'snapshot_date', 'snapshot_id': 'snapshot_id', 'familia': 'producto', 'http_status': 'http_status', 'raw': 'raw_rows', 'processed': 'processed_rows', 'duplicados': 'exact_duplicates_removed', 'error': 'error'}


,snapshot_date,snapshot_id,run_at,producto,http_status,raw_rows,exact_duplicates_removed,processed_rows,entidades,fecha_info_min,fecha_info_max,sha256,previous_sha256,changed_vs_previous,raw_path,flattened_path,processed_path,error
0,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,200,321,51,270,30,2017-04-06T00:00:00,2026-09-07T00:00:00,d9d3028285c7f650ffe1bf3680ae0174e3a13c8ee76d04db7b44c707dd766963,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\hipotecarios.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\hipotecarios.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\hipotecarios.csv,NaN
1,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,personales,200,2380,98,2282,291,2020-05-04T00:00:00,2026-09-07T00:00:00,745d5b2593d73bb993076ba3b1af75088cea66d664e67c2f24d642e6982fed21,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\personales.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\personales.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\personales.csv,NaN
2,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,200,493,2,491,48,2017-04-06T00:00:00,2026-09-07T00:00:00,70f915558e14af2ed5b88ba9025b03659df697a4d4621261515f18b349edf7ae,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\prendarios.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\prendarios.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\prendarios.csv,NaN
3,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,tarjetas,200,541,0,541,137,2019-08-27T00:00:00,2026-09-07T00:00:00,875d3e24f710a07808ac63364fc9f6aa751c9e58a41cde3ebc4d910de514e639,NaN,NaN,data_b\snapshots\2026-09-07\20260907_212935_569641\raw\tarjetas.json,data_b\snapshots\2026-09-07\20260907_212935_569641\flattened\tarjetas.csv,data_b\snapshots\2026-09-07\20260907_212935_569641\processed\tarjetas.csv,NaN
4,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,200,319,51,268,30,2017-04-06T00:00:00,2026-09-08T00:00:00,9bbce0150dd2565b74e1298827b1ab6ba06a498c7c4c859ca0c66f90bb2f0e29,d9d3028285c7f650ffe1bf3680ae0174e3a13c8ee76d04db7b44c707dd766963,True,data_b\snapshots\2026-09-08\20260908_224655_838862\raw\hipotecarios.json,data_b\snapshots\2026-09-08\20260908_224655_838862\flattened\hipotecarios.csv,data_b\snapshots\2026-09-08\20260908_224655_838862\processed\hipotecarios.csv,NaN
5,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,personales,200,2380,98,2282,291,2020-05-04T00:00:00,2026-09-08T00:00:00,b73628b79c4557818b4775b39f18d148c9af8a13f9aebb7ab2508d271764c969,745d5b2593d73bb993076ba3b1af75088cea66d664e67c2f24d642e6982fed21,True,data_b\snapshots\2026-09-08\20260908_224655_838862\raw\personales.json,data_b\snapshots\2026-09-08\20260908_224655_838862\flattened\personales.csv,data_b\snapshots\2026-09-08\20260908_224655_838862\processed\personales.csv,NaN
6,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,prendarios,200,493,2,491,48,2017-04-06T00:00:00,2026-09-08T00:00:00,a956a66163aac98ef36c5121f08b3a6272149dcf97535cc6ea087c37322a5cc1,70f915558e14af2ed5b88ba9025b03659df697a4d4621261515f18b349edf7ae,True,data_b\snapshots\2026-09-08\20260908_224655_838862\raw\prendarios.json,data_b\snapshots\2026-09-08\20260908_224655_838862\flattened\prendarios.csv,data_b\snapshots\2026-09-08\20260908_224655_838862\processed\prendarios.csv,NaN
7,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,tarjetas,200,541,0,541,137,2019-08-27T00:00:00,2026-09-08T00:00:00,913492043d7d2c5af499b4a7fe6fa12f52aa78e92cd7daf24bf782d3968e588f,875d3e24f710a07808ac63364fc9f6aa751c9e58a41cde3ebc4d910de514e639,True,data_b\snapshots\2026-09-08\20260908_224655_838862\raw\tarjetas.json,data_b\snapshots\2026-09-08\20260908_224655_838862\flattened\tarjetas.csv,data_b\snapshots\2026-09-08\20260908_224655_838862\processed\tarjetas.csv,NaN
8,2026-09-09

In [4]:

# Construir inventario resumen a partir del manifest.
m = manifest.copy()

# Si el nombre de duplicados no fue detectado, derivarlo cuando sea posible.
if dup_col is None and raw_col and proc_col:
    m["_duplicates_derived"] = m[raw_col] - m[proc_col]
    dup_col = "_duplicates_derived"

summary_rows = []
for (d, sid), g in m.groupby([date_col, sid_col], dropna=False):
    r = {"date": d, "snapshot_id": sid, "families": ", ".join(sorted(g[fam_col].astype(str).unique()))}
    for fam in FAMILIES:
        gf = g[g[fam_col].astype(str).str.lower() == fam]
        if len(gf):
            x = gf.iloc[0]
            r[f"{fam}_raw"] = x[raw_col] if raw_col else np.nan
            r[f"{fam}_processed"] = x[proc_col] if proc_col else np.nan
            r[f"{fam}_duplicates"] = x[dup_col] if dup_col else np.nan
            r[f"{fam}_http"] = x[status_col] if status_col else np.nan
            if err_col:
                r[f"{fam}_error"] = x[err_col]
    summary_rows.append(r)

manifest_inventory = pd.DataFrame(summary_rows).sort_values(["date","snapshot_id"])
display(manifest_inventory)

if status_col:
    print("\nHTTP status:")
    display(m[status_col].value_counts(dropna=False).rename_axis("status").to_frame("count"))

if err_col:
    print("\nErrores no vacíos:")
    err = m[err_col].dropna().astype(str)
    err = err[err.str.strip() != ""]
    display(err.to_frame("error"))
else:
    print("\nEl manifest no contiene una columna de error identificable con el detector.")


,date,snapshot_id,families,hipotecarios_raw,hipotecarios_processed,hipotecarios_duplicates,hipotecarios_http,hipotecarios_error,personales_raw,personales_processed,personales_duplicates,personales_http,personales_error,prendarios_raw,prendarios_processed,prendarios_duplicates,prendarios_http,prendarios_error,tarjetas_raw,tarjetas_processed,tarjetas_duplicates,tarjetas_http,tarjetas_error
0,2026-09-07,20260907_212935_569641,"hipotecarios, personales, prendarios, tarjetas",321,270,51,200,NaN,2380,2282,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
1,2026-09-08,20260908_224655_838862,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2380,2282,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
2,2026-09-09,20260909_230424_341254,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2383,2285,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
3,2026-09-10,20260910_221306_387342,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2382,2284,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
4,2026-09-11,20260911_215454_602912,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2382,2284,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
5,2026-09-12,20260912_192218_082470,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2382,2284,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
6,2026-09-13,20260913_220415_571963,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2382,2284,98,200,NaN,493,491,2,200,NaN,541,541,0,200,NaN
7,2026-09-14,20260914_231738_476390,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2384,2286,98,200,NaN,494,492,2,200,NaN,541,541,0,200,NaN
8,2026-09-15,20260915_230313_952585,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2354,2256,98,200,NaN,494,492,2,200,NaN,541,541,0,200,NaN
9,2026-09-16,20260916_235331_528166,"hipotecarios, personales, prendarios, tarjetas",319,268,51,200,NaN,2376,2278,98,200,NaN,494,492,2,200,NaN,541,541,0,200,NaN



HTTP status:


,count
status,
200,48



Errores no vacíos:


,error



## 3. Corridas físicas no registradas en el manifest

Esto permite detectar, por ejemplo, corridas interrumpidas o parciales.  
No se las califica automáticamente como “dato malo”: se documenta su estado.


In [5]:
registered = set(zip(manifest[date_col].astype(str), manifest[sid_col].astype(str)))
physical = set(zip(physical_inventory.snapshot_date_dir.astype(str), physical_inventory.snapshot_id.astype(str)))

unregistered = physical - registered
missing_physically = registered - physical

print("Directorios físicos no registrados en manifest:", unregistered)
print("Snapshots registrados en manifest sin directorio físico:", missing_physically)

display(
    physical_inventory[
        physical_inventory.apply(
            lambda r: (str(r.snapshot_date_dir), str(r.snapshot_id)) in unregistered, axis=1
        )
    ][["snapshot_date_dir","snapshot_id","n_raw_families","n_flat_families","n_proc_families","all_products","metadata"]]
)


Directorios físicos no registrados en manifest: set()
Snapshots registrados en manifest sin directorio físico: set()


,snapshot_date_dir,snapshot_id,n_raw_families,n_flat_families,n_proc_families,all_products,metadata


In [6]:
# Inspección detallada de cualquier corrida física no registrada.
for date, sid in sorted(unregistered):
    base = SNAPSHOT_ROOT / date / sid
    print("\n", "="*100)
    print(date, sid)
    present = sorted(
        path.relative_to(base).as_posix()
        for path in base.rglob("*")
        if path.is_file()
    )
    for name in present:
        print(" -", name)

    for fam in FAMILIES:
        raw_path = base / "raw" / f"{fam}.json"
        flat_path = base / "flattened" / f"{fam}.csv"
        proc_path = base / "processed" / f"{fam}.csv"
        if raw_path.is_file():
            with raw_path.open(encoding="utf-8") as handle:
                obj = json.load(handle)
            print(f"\n{fam}: raw status={obj.get('status')} results={len(obj.get('results', []))}")
        if flat_path.is_file():
            print(f"{fam}: flattened rows={len(pd.read_csv(flat_path))}")
        if proc_path.is_file():
            print(f"{fam}: processed rows={len(pd.read_csv(proc_path))}")



## 4. Cobertura temporal

Se distinguen:

- **fecha física existente**;
- **snapshot completo**;
- **cantidad de snapshots completos por día**.

Esto es importante porque una fecha puede tener más de una corrida válida.


In [7]:

complete = physical_inventory[physical_inventory.physically_complete].copy()
coverage = (
    complete.groupby("snapshot_date_dir")
    .agg(n_complete_snapshots=("snapshot_id","nunique"),
         snapshot_ids=("snapshot_id", lambda s: " | ".join(sorted(s))))
    .reset_index()
)
coverage["snapshot_date_dir"] = pd.to_datetime(coverage["snapshot_date_dir"])
all_days = pd.date_range(coverage.snapshot_date_dir.min(), coverage.snapshot_date_dir.max(), freq="D")
missing_days = sorted(set(all_days.date) - set(coverage.snapshot_date_dir.dt.date))

display(coverage)
print("Fechas faltantes entre mínimo y máximo:", missing_days)


,snapshot_date_dir,n_complete_snapshots,snapshot_ids
0,2026-09-07,1,20260907_212935_569641
1,2026-09-08,1,20260908_224655_838862
2,2026-09-09,1,20260909_230424_341254
3,2026-09-10,1,20260910_221306_387342
4,2026-09-11,1,20260911_215454_602912
5,2026-09-12,1,20260912_192218_082470
6,2026-09-13,1,20260913_220415_571963
7,2026-09-14,1,20260914_231738_476390
8,2026-09-15,1,20260915_230313_952585
9,2026-09-16,1,20260916_235331_528166


Fechas faltantes entre mínimo y máximo: []



## 5. Estructura de una corrida completa

Se verifica empíricamente la arquitectura:

`raw/` → respuesta de API  
`flattened/` → una fila por resultado + metadatos de snapshot  
`processed/` → deduplicación + derivaciones + claves + flags

Se toma el snapshot completo más reciente sólo para mostrar la estructura, y luego se valida la estabilidad contra **todos** los snapshots completos.


In [8]:
latest_complete = complete.sort_values(["snapshot_date_dir","snapshot_id"]).iloc[-1]
LATEST_DATE = latest_complete.snapshot_date_dir
LATEST_SID = latest_complete.snapshot_id
LATEST_BASE = SNAPSHOT_ROOT / str(LATEST_DATE) / str(LATEST_SID)

print("Snapshot usado como ejemplo:", LATEST_DATE, LATEST_SID)

example_files = sorted(
    path.relative_to(LATEST_BASE).as_posix()
    for path in LATEST_BASE.rglob("*")
    if path.is_file()
)
print("\n".join(example_files))


Snapshot usado como ejemplo: 2026-09-18 20260918_225718_879151
flattened/hipotecarios.csv
flattened/personales.csv
flattened/prendarios.csv
flattened/tarjetas.csv
processed/all_products.csv
processed/hipotecarios.csv
processed/personales.csv
processed/prendarios.csv
processed/tarjetas.csv
raw/hipotecarios.json
raw/personales.json
raw/prendarios.json
raw/tarjetas.json
snapshot_metadata.json



## 6. Forma superior de los JSON raw y consistencia de claves

Se comprueba:

- claves de nivel superior;
- `status`;
- cantidad de `results`;
- cantidad de variantes de schema de objetos dentro de cada familia;
- campos presentes en todos los objetos raw.

**Nota metodológica:** “presente en todos los objetos observados” no equivale a “obligatorio según contrato de API”.


In [9]:
raw_schema_summary = []

for fam in FAMILIES:
    path = LATEST_BASE / "raw" / f"{fam}.json"
    with path.open(encoding="utf-8") as handle:
        obj = json.load(handle)
    results = obj.get("results", [])
    schemas = Counter(tuple(sorted(r.keys())) for r in results)
    intersection = set.intersection(*(set(r.keys()) for r in results)) if results else set()
    union = set.union(*(set(r.keys()) for r in results)) if results else set()

    raw_schema_summary.append({
        "familia": fam,
        "top_level_keys": sorted(obj.keys()),
        "status": obj.get("status"),
        "n_results": len(results),
        "n_schema_variants": len(schemas),
        "n_union_fields": len(union),
        "n_fields_present_in_every_object": len(intersection),
        "all_objects_same_keys": len(schemas) == 1
    })

display(pd.DataFrame(raw_schema_summary))


,familia,top_level_keys,status,n_results,n_schema_variants,n_union_fields,n_fields_present_in_every_object,all_objects_same_keys
0,hipotecarios,"[results, status]",200,272,1,22,22,True
1,personales,"[results, status]",200,2390,1,21,21,True
2,prendarios,"[results, status]",200,494,1,23,23,True
3,tarjetas,"[results, status]",200,537,1,15,15,True



## 7. Schema por familia y capa

El bloque siguiente muestra:

- cantidad de columnas;
- nombres;
- dtype inferido por pandas.

Después se verifica si existe **schema drift físico** entre snapshots completos.


In [10]:
schemas_latest = {}
for fam in FAMILIES:
    schemas_latest[fam] = {}
    for layer, rel in [
        ("flattened", Path("flattened") / f"{fam}.csv"),
        ("processed", Path("processed") / f"{fam}.csv"),
    ]:
        df = pd.read_csv(LATEST_BASE / rel)
        schemas_latest[fam][layer] = {
            "columns": list(df.columns),
            "dtypes": df.dtypes.astype(str).to_dict(),
            "n_rows": len(df)
        }
        print("\n", "="*100)
        print(f"{fam.upper()} — {layer} — rows={len(df)} cols={df.shape[1]}")
        display(pd.DataFrame({
            "column": df.columns,
            "dtype_inferred": df.dtypes.astype(str).values
        }))



HIPOTECARIOS — flattened — rows=272 cols=26


,column,dtype_inferred
0,snapshot_date,str
1,snapshot_id,str
2,snapshot_run_at,str
3,producto_snapshot,str
4,relacionMontoTasacion,float64
5,destinoFondos,str
6,denominacion,str
7,montoMaximoOtorgable,float64
8,plazoMaximoOtorgable,float64
9,ingresoMinimoMensual,float64



HIPOTECARIOS — processed — rows=272 cols=39


,column,dtype_inferred
0,snapshot_date,str
1,snapshot_id,str
2,snapshot_run_at,str
3,producto_snapshot,str
4,relacionMontoTasacion,float64
5,destinoFondos,str
6,denominacion,str
7,montoMaximoOtorgable,float64
8,plazoMaximoOtorgable,float64
9,ingresoMinimoMensual,float64



PERSONALES — flattened — rows=2390 cols=25


,column,dtype_inferred
0,snapshot_date,str
1,snapshot_id,str
2,snapshot_run_at,str
3,producto_snapshot,str
4,montoMinimoOtorgable,float64
5,denominacion,str
6,montoMaximoOtorgable,float64
7,plazoMaximoOtorgable,float64
8,ingresoMinimoMensual,float64
9,antiguedadLaboralMinimaMeses,float64


... (6 outputs left out)


TARJETAS — flattened — rows=537 cols=19


,column,dtype_inferred
0,snapshot_date,str
1,snapshot_id,str
2,snapshot_run_at,str
3,producto_snapshot,str
4,comisionMaximaAdministracionMantenimiento,float64
5,comisionMaximaRenovacion,float64
6,tasaEfectivaAnualMaximaFinanciacion,float64
7,tasaEfectivaAnualMaximaAdelantoEfectivo,float64
8,ingresoMinimoMensual,float64
9,antiguedadLaboralMinimaMeses,float64



TARJETAS — processed — rows=537 cols=28


,column,dtype_inferred
0,snapshot_date,str
1,snapshot_id,str
2,snapshot_run_at,str
3,producto_snapshot,str
4,comisionMaximaAdministracionMantenimiento,float64
5,comisionMaximaRenovacion,float64
6,tasaEfectivaAnualMaximaFinanciacion,float64
7,tasaEfectivaAnualMaximaAdelantoEfectivo,float64
8,ingresoMinimoMensual,float64
9,antiguedadLaboralMinimaMeses,float64


In [11]:
# Validación de drift de headers y dtypes observados.
schema_records = []
for _, snap in complete.iterrows():
    d, sid = snap.snapshot_date_dir, snap.snapshot_id
    base = SNAPSHOT_ROOT / str(d) / str(sid)
    for fam in FAMILIES:
        for layer in ["flattened","processed"]:
            df = pd.read_csv(base / layer / f"{fam}.csv")
            schema_records.append({
                "date": d,
                "snapshot_id": sid,
                "familia": fam,
                "layer": layer,
                "header_signature": tuple(df.columns),
                "dtype_signature": tuple((c, str(t)) for c,t in df.dtypes.items())
            })

schema_df = pd.DataFrame(schema_records)
schema_stability = (
    schema_df.groupby(["familia","layer"])
    .agg(
        distinct_headers=("header_signature","nunique"),
        distinct_dtype_signatures=("dtype_signature","nunique")
    )
    .reset_index()
)
display(schema_stability)


,familia,layer,distinct_headers,distinct_dtype_signatures
0,hipotecarios,flattened,1,1
1,hipotecarios,processed,1,1
2,personales,flattened,1,1
3,personales,processed,1,1
4,prendarios,flattened,1,1
5,prendarios,processed,1,1
6,tarjetas,flattened,1,1
7,tarjetas,processed,1,1


In [12]:
# `complete` ya contiene sólo los snapshots completos incluidos en EDA_CUTOFF_DATE.
raw_schema_records = []
for _, snap in complete.iterrows():
    snapshot_date = str(snap["snapshot_date_dir"])
    snapshot_id = str(snap["snapshot_id"])
    snapshot_base = SNAPSHOT_ROOT / snapshot_date / snapshot_id

    for familia in FAMILIES:
        raw_path = snapshot_base / "raw" / f"{familia}.json"
        with raw_path.open(encoding="utf-8") as handle:
            obj = json.load(handle)

        results = obj.get("results", [])
        result_schema_signature = tuple(
            sorted(set(tuple(sorted(row.keys())) for row in results))
        )
        result_key_sets = [set(row.keys()) for row in results]
        union_fields = tuple(sorted(set().union(*result_key_sets))) if result_key_sets else ()
        intersection_fields = (
            tuple(sorted(set.intersection(*result_key_sets)))
            if result_key_sets
            else ()
        )

        raw_schema_records.append({
            "snapshot_date": snapshot_date,
            "snapshot_id": snapshot_id,
            "familia": familia,
            "top_level_signature": tuple(sorted(obj.keys())),
            "n_result_schema_variants": len(result_schema_signature),
            "union_fields": union_fields,
            "intersection_fields": intersection_fields,
            "schema_signature": result_schema_signature,
        })

raw_schema_stability = pd.DataFrame(raw_schema_records)
display(raw_schema_stability)

raw_schema_summary = (
    raw_schema_stability
    .groupby("familia")
    .agg(
        distinct_result_schemas=("schema_signature", "nunique"),
        distinct_top_level_schemas=("top_level_signature", "nunique"),
    )
    .reset_index()
)

display(raw_schema_summary)


,snapshot_date,snapshot_id,familia,top_level_signature,n_result_schema_variants,union_fields,intersection_fields,schema_signature
0,2026-09-07,20260907_212935_569641,hipotecarios,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
1,2026-09-07,20260907_212935_569641,personales,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
2,2026-09-07,20260907_212935_569641,prendarios,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
3,2026-09-07,20260907_212935_569641,tarjetas,"(results, status)",1,"(antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, d...","(antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, d...","((antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, ..."
4,2026-09-08,20260908_224655_838862,hipotecarios,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
5,2026-09-08,20260908_224655_838862,personales,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
6,2026-09-08,20260908_224655_838862,prendarios,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfecti..."
7,2026-09-08,20260908_224655_838862,tarjetas,"(results, status)",1,"(antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, d...","(antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, d...","((antiguedadLaboralMinimaMeses, codigoEntidad, comisionMaximaAdministracionMantenimiento, comisionMaximaRenovacion, ..."
8,2026-09-09,20260909_230424_341254,hipotecarios,"(results, status)",1,"(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","(antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAnticipada, codigoEntidad, costoFinancieroEfectiv...","((antiguedadLaboralMinimaMeses, beneficiario, cargoMaximoCancelacionAntici

,familia,distinct_result_schemas,distinct_top_level_schemas
0,hipotecarios,1,1
1,personales,1,1
2,prendarios,1,1
3,tarjetas,1,1



## 8. Diferencias de schema entre familias

Primero se compara el schema **raw/flattened de negocio**, excluyendo metadatos técnicos.  
Luego se compara el schema processed para identificar campos derivados.


In [13]:

META_FLAT = {"snapshot_date","snapshot_id","snapshot_run_at","producto_snapshot"}

flat_sets = {}
proc_sets = {}
for fam in FAMILIES:
    flat_sets[fam] = set(schemas_latest[fam]["flattened"]["columns"]) - META_FLAT
    proc_sets[fam] = set(schemas_latest[fam]["processed"]["columns"])

all_flat = sorted(set.union(*flat_sets.values()))
matrix = pd.DataFrame(index=all_flat)
for fam in FAMILIES:
    matrix[fam] = [c in flat_sets[fam] for c in all_flat]

display(matrix)

common_raw_business = set.intersection(*flat_sets.values())
print("Campos de negocio comunes a las cuatro familias:")
print(sorted(common_raw_business))

print("\nCampos exclusivos o no universales:")
display(matrix[matrix.nunique(axis=1) > 1])


,hipotecarios,personales,prendarios,tarjetas
antiguedadLaboralMinimaMeses,True,True,True,True
beneficiario,True,True,True,False
cargoMaximoCancelacionAnticipada,True,True,True,False
codigoEntidad,True,True,True,True
comisionMaximaAdministracionMantenimiento,False,False,False,True
comisionMaximaRenovacion,False,False,False,True
costoFinancieroEfectivoTotalMaximo,True,True,True,False
cuotaInicial,True,True,True,False
denominacion,True,True,True,False
descripcionEntidad,True,True,True,True


Campos de negocio comunes a las cuatro familias:
['antiguedadLaboralMinimaMeses', 'codigoEntidad', 'descripcionEntidad', 'edadMaximaSolicitada', 'fechaInformacion', 'ingresoMinimoMensual', 'masInformacion', 'nombreCompleto', 'nombreCorto', 'territorioValidez']

Campos exclusivos o no universales:


,hipotecarios,personales,prendarios,tarjetas
beneficiario,True,True,True,False
cargoMaximoCancelacionAnticipada,True,True,True,False
comisionMaximaAdministracionMantenimiento,False,False,False,True
comisionMaximaRenovacion,False,False,False,True
costoFinancieroEfectivoTotalMaximo,True,True,True,False
cuotaInicial,True,True,True,False
denominacion,True,True,True,False
destinoFondos,True,False,True,False
montoMaximoOtorgable,True,True,True,False
montoMinimoOtorgable,False,True,True,False



## 9. Diccionario estructural

La descripción semántica de los campos se basa en:

- nombre del campo;
- valores observados;
- su posición en el pipeline.

No se presenta como contrato oficial de la API. Para cada campo se identifica además su origen:

- `BCRA/raw`;
- `metadata pipeline`;
- `derivada pipeline`;
- `flag pipeline`.


In [14]:

SEMANTICS = {
    "snapshot_date": "Fecha calendario de la captura.",
    "snapshot_id": "Identificador único de la ejecución/captura.",
    "snapshot_run_at": "Timestamp de ejecución del extractor.",
    "producto_snapshot": "Familia de producto asociada al archivo/fila.",
    "codigoEntidad": "Código identificador de la entidad informado por la fuente.",
    "descripcionEntidad": "Nombre/descripción de la entidad.",
    "fechaInformacion": "Fecha de vigencia/actualización informada para la oferta.",
    "nombreCompleto": "Nombre completo del producto.",
    "nombreCorto": "Nombre abreviado del producto.",
    "territorioValidez": "Territorio o alcance geográfico de validez informado.",
    "masInformacion": "Texto libre con condiciones o información adicional.",
    "denominacion": "Denominación/unidad o modalidad monetaria del producto (p.ej. Pesos/UVA).",
    "montoMinimoOtorgable": "Monto mínimo otorgable informado.",
    "montoMaximoOtorgable": "Monto máximo otorgable informado.",
    "plazoMaximoOtorgable": "Plazo máximo otorgable informado.",
    "ingresoMinimoMensual": "Ingreso mínimo mensual requerido/informado.",
    "antiguedadLaboralMinimaMeses": "Antigüedad laboral mínima requerida, en meses.",
    "edadMaximaSolicitada": "Edad máxima informada para solicitantes.",
    "relacionCuotaIngreso": "Relación cuota/ingreso publicada por la fuente.",
    "relacionMontoTasacion": "Relación monto/tasación publicada por la fuente.",
    "beneficiario": "Segmento o condición de beneficiario.",
    "cargoMaximoCancelacionAnticipada": "Cargo máximo informado por cancelación anticipada.",
    "tasaEfectivaAnualMaxima": "TEA máxima informada.",
    "tipoTasa": "Tipo/modalidad de tasa informada.",
    "costoFinancieroEfectivoTotalMaximo": "CFTEA máximo informado.",
    "cuotaInicial": "Cuota inicial informada.",
    "destinoFondos": "Destino declarado de los fondos.",
    "comisionMaximaAdministracionMantenimiento": "Comisión máxima de administración/mantenimiento de tarjeta.",
    "comisionMaximaRenovacion": "Comisión máxima de renovación de tarjeta.",
    "tasaEfectivaAnualMaximaFinanciacion": "TEA máxima para financiación de tarjeta.",
    "tasaEfectivaAnualMaximaAdelantoEfectivo": "TEA máxima para adelanto de efectivo.",
    "segmento": "Segmento/categoría de tarjeta.",
    "information_age_days": "Edad en días de fechaInformacion respecto de la captura.",
    "information_age_bucket": "Bucket derivado de antigüedad de información.",
    "product_family_key": "Clave derivada para agrupar una familia lógica de producto.",
    "offer_variant_key": "Clave derivada para distinguir variantes/ofertas dentro de una familia lógica.",
    "product_key_version": "Versión del algoritmo de claves de producto.",
    "relacionCuotaIngreso_raw": "Copia auditiva del valor original de relación cuota/ingreso.",
    "relacionCuotaIngreso_pct": "Versión normalizada a porcentaje de relación cuota/ingreso.",
    "relacionMontoTasacion_raw": "Copia auditiva del valor original de relación monto/tasación.",
    "relacionMontoTasacion_pct": "Versión normalizada a porcentaje de relación monto/tasación.",
}

def origin_of(col):
    if col in META_FLAT:
        return "metadata pipeline"
    if col.endswith("__extreme_gt_1000") or col.endswith("__negative_or_sentinel"):
        return "flag pipeline"
    if col in {"information_age_days","information_age_bucket","product_family_key",
               "offer_variant_key","product_key_version","relacionCuotaIngreso_raw",
               "relacionCuotaIngreso_pct","relacionMontoTasacion_raw","relacionMontoTasacion_pct"}:
        return "derivada pipeline"
    return "BCRA/raw"

dictionary_rows = []
for fam in FAMILIES:
    cols = schemas_latest[fam]["processed"]["columns"]
    dtypes = schemas_latest[fam]["processed"]["dtypes"]
    for c in cols:
        dictionary_rows.append({
            "familia": fam,
            "columna": c,
            "origen": origin_of(c),
            "dtype_observado": dtypes[c],
            "semantica": SEMANTICS.get(c, "Flag/atributo técnico derivado; revisar implementación del extractor.")
        })

dictionary_df = pd.DataFrame(dictionary_rows)
display(dictionary_df)


,familia,columna,origen,dtype_observado,semantica
0,hipotecarios,snapshot_date,metadata pipeline,str,Fecha calendario de la captura.
1,hipotecarios,snapshot_id,metadata pipeline,str,Identificador único de la ejecución/captura.
2,hipotecarios,snapshot_run_at,metadata pipeline,str,Timestamp de ejecución del extractor.
3,hipotecarios,producto_snapshot,metadata pipeline,str,Familia de producto asociada al archivo/fila.
4,hipotecarios,relacionMontoTasacion,BCRA/raw,float64,Relación monto/tasación publicada por la fuente.
5,hipotecarios,destinoFondos,BCRA/raw,str,Destino declarado de los fondos.
6,hipotecarios,denominacion,BCRA/raw,str,Denominación/unidad o modalidad monetaria del producto (p.ej. Pesos/UVA).
7,hipotecarios,montoMaximoOtorgable,BCRA/raw,float64,Monto máximo otorgable informado.
8,hipotecarios,plazoMaximoOtorgable,BCRA/raw,float64,Plazo máximo otorgable informado.
9,hipotecarios,ingresoMinimoMensual,BCRA/raw,float64,Ingreso mínimo mensual requerido/informado.


## 10. Presencia, nulos y columnas completamente vacías

Esto responde de forma **observacional**, no contractual:

- qué columnas existen;
- cuántos valores no nulos tienen;
- cuáles están completas;
- cuáles están completamente vacías.

Se calcula sobre todos los snapshots completos.

### Resumen del hallazgo

No aparece ninguna columna completamente vacía en las familias analizadas. Esto es positivo desde el punto de vista de presencia estructural.

Sin embargo, `masInformacion` presenta una nulabilidad naturalmente alta y heterogénea:

| Familia | Nulos en `masInformacion` |
|---|---:|
| Hipotecarios | 29,8% |
| Personales | 74,0% |
| Prendarios | 28,3% |
| Tarjetas | 53,4% |

También aparecen registros de producto muy incompletos que se repiten entre snapshots. En hipotecarios y prendarios se observa, por ejemplo, `TARJETA GRUPAR S.A.` con prácticamente todos los atributos del producto vacíos y sólo la entidad junto con `fechaInformacion=2017-04-06`. En personales aparece un patrón similar en registros como `FINANCE GROUP SRL`.

Estos casos no se eliminan ni se descartan en esta etapa. La pregunta concreta para el EDA de calidad es:

> ¿La API incluye entidades o fichas sin oferta efectiva como registros de producto?

Esto debe distinguirse de los nulos dispersos esperables en atributos opcionales.


In [15]:
null_records = []
for _, snap in complete.iterrows():
    d, sid = snap.snapshot_date_dir, snap.snapshot_id
    base = SNAPSHOT_ROOT / str(d) / str(sid)
    for fam in FAMILIES:
        df = pd.read_csv(base / "processed" / f"{fam}.csv")
        for c in df.columns:
            null_records.append({
                "date": d,
                "snapshot_id": sid,
                "familia": fam,
                "column": c,
                "rows": len(df),
                "non_null": int(df[c].notna().sum()),
                "nulls": int(df[c].isna().sum()),
                "null_pct": float(df[c].isna().mean() * 100),
                "completely_empty": bool(df[c].isna().all())
            })

null_df = pd.DataFrame(null_records)

overall_null = (
    null_df.groupby(["familia","column"])
    .agg(
        snapshots=("snapshot_id","nunique"),
        total_rows=("rows","sum"),
        total_non_null=("non_null","sum"),
        total_nulls=("nulls","sum"),
        max_null_pct_snapshot=("null_pct","max"),
        ever_completely_empty=("completely_empty","max")
    )
    .reset_index()
)
overall_null["overall_null_pct"] = 100 * overall_null.total_nulls / overall_null.total_rows

for fam in FAMILIES:
    print("\n", "="*100)
    print(fam.upper())
    display(
        overall_null[overall_null.familia == fam]
        .sort_values(["overall_null_pct","column"], ascending=[False,True])
    )



HIPOTECARIOS


,familia,column,snapshots,total_rows,total_non_null,total_nulls,max_null_pct_snapshot,ever_completely_empty,overall_null_pct
16,hipotecarios,masInformacion,12,3223,2263,960,29.850746,False,29.785914
0,hipotecarios,antiguedadLaboralMinimaMeses,12,3223,3211,12,0.373134,False,0.372324
1,hipotecarios,beneficiario,12,3223,3211,12,0.373134,False,0.372324
2,hipotecarios,cargoMaximoCancelacionAnticipada,12,3223,3211,12,0.373134,False,0.372324
4,hipotecarios,costoFinancieroEfectivoTotalMaximo,12,3223,3211,12,0.373134,False,0.372324
7,hipotecarios,cuotaInicial,12,3223,3211,12,0.373134,False,0.372324
8,hipotecarios,denominacion,12,3223,3211,12,0.373134,False,0.372324
10,hipotecarios,destinoFondos,12,3223,3211,12,0.373134,False,0.372324
11,hipotecarios,edadMaximaSolicitada,12,3223,3211,12,0.373134,False,0.372324
15,hipotecarios,ingresoMinimoMensual,12,3223,3211,12,0.373134,False,0.372324



PERSONALES


,familia,column,snapshots,total_rows,total_non_null,total_nulls,max_null_pct_snapshot,ever_completely_empty,overall_null_pct
54,personales,masInformacion,12,27390,7077,20313,74.825480,False,74.162103
58,personales,nombreCorto,12,27390,27150,240,0.886525,False,0.876232
47,personales,denominacion,12,27390,27294,96,0.354610,False,0.350493
53,personales,ingresoMinimoMensual,12,27390,27342,48,0.177305,False,0.175246
64,personales,relacionCuotaIngreso,12,27390,27342,48,0.177305,False,0.175246
65,personales,relacionCuotaIngreso_pct,12,27390,27342,48,0.177305,False,0.175246
66,personales,relacionCuotaIngreso_raw,12,27390,27342,48,0.177305,False,0.175246
41,personales,cargoMaximoCancelacionAnticipada,12,27390,27354,36,0.132979,False,0.131435
74,personales,tipoTasa,12,27390,27354,36,0.132979,False,0.131435
39,personales,antiguedadLaboralMinimaMeses,12,27390,27366,24,0.088652,False,0.087623



PRENDARIOS


,familia,column,snapshots,total_rows,total_non_null,total_nulls,max_null_pct_snapshot,ever_completely_empty,overall_null_pct
91,prendarios,masInformacion,12,5897,4208,1689,31.707317,False,28.641682
97,prendarios,plazoMaximoOtorgable,12,5897,5705,192,3.258656,False,3.255893
75,prendarios,antiguedadLaboralMinimaMeses,12,5897,5885,12,0.203666,False,0.203493
76,prendarios,beneficiario,12,5897,5885,12,0.203666,False,0.203493
77,prendarios,cargoMaximoCancelacionAnticipada,12,5897,5885,12,0.203666,False,0.203493
79,prendarios,costoFinancieroEfectivoTotalMaximo,12,5897,5885,12,0.203666,False,0.203493
82,prendarios,cuotaInicial,12,5897,5885,12,0.203666,False,0.203493
83,prendarios,denominacion,12,5897,5885,12,0.203666,False,0.203493
85,prendarios,destinoFondos,12,5897,5885,12,0.203666,False,0.203493
86,prendarios,edadMaximaSolicitada,12,5897,5885,12,0.203666,False,0.203493



TARJETAS


,familia,column,snapshots,total_rows,total_non_null,total_nulls,max_null_pct_snapshot,ever_completely_empty,overall_null_pct
125,tarjetas,masInformacion,12,6484,3024,3460,53.419593,False,53.362122
136,tarjetas,tasaEfectivaAnualMaximaAdelantoEfectivo,12,6484,6088,396,6.145251,False,6.107341
118,tarjetas,comisionMaximaRenovacion,12,6484,6376,108,1.675978,False,1.665638
132,tarjetas,segmento,12,6484,6388,96,1.489758,False,1.480568
115,tarjetas,antiguedadLaboralMinimaMeses,12,6484,6448,36,0.558659,False,0.555213
124,tarjetas,ingresoMinimoMensual,12,6484,6448,36,0.558659,False,0.555213
117,tarjetas,comisionMaximaAdministracionMantenimiento,12,6484,6460,24,0.372439,False,0.370142
120,tarjetas,edadMaximaSolicitada,12,6484,6460,24,0.372439,False,0.370142
126,tarjetas,nombreCompleto,12,6484,6460,24,0.372439,False,0.370142
139,tarjetas,tasaEfectivaAnualMaximaFinanciacion,12,6484,6460,24,0.372439,False,0.370142



## 11. ¿Qué campos son “obligatorios”?

Se separan dos niveles:

### A. Presencia estructural observada
Campo incluido como clave/columna en todos los registros o snapshots observados.

### B. Obligatoriedad contractual
Sólo puede afirmarse con el schema/OpenAPI/documentación normativa de la API.

Este ZIP permite responder **A**, pero no prueba **B**.


In [16]:

# Campos de processed que nunca fueron nulos en ningún snapshot completo.
observational_required = (
    overall_null[overall_null.total_nulls == 0]
    .groupby("familia")["column"]
    .apply(list)
)

for fam, cols in observational_required.items():
    print(f"\n{fam}: {len(cols)} columnas sin nulos observados")
    print(cols)



hipotecarios: 16 columnas sin nulos observados
['codigoEntidad', 'costoFinancieroEfectivoTotalMaximo__extreme_gt_1000', 'costoFinancieroEfectivoTotalMaximo__negative_or_sentinel', 'descripcionEntidad', 'fechaInformacion', 'information_age_bucket', 'information_age_days', 'offer_variant_key', 'product_family_key', 'product_key_version', 'producto_snapshot', 'snapshot_date', 'snapshot_id', 'snapshot_run_at', 'tasaEfectivaAnualMaxima__extreme_gt_1000', 'tasaEfectivaAnualMaxima__negative_or_sentinel']

personales: 16 columnas sin nulos observados
['codigoEntidad', 'costoFinancieroEfectivoTotalMaximo__extreme_gt_1000', 'costoFinancieroEfectivoTotalMaximo__negative_or_sentinel', 'descripcionEntidad', 'fechaInformacion', 'information_age_bucket', 'information_age_days', 'offer_variant_key', 'product_family_key', 'product_key_version', 'producto_snapshot', 'snapshot_date', 'snapshot_id', 'snapshot_run_at', 'tasaEfectivaAnualMaxima__extreme_gt_1000', 'tasaEfectivaAnualMaxima__negative_or_senti


## 12. Tipos físicos y divergencias entre familias

Los CSV no almacenan tipos lógicos fuertes; pandas los infiere.  
Se buscan campos del mismo nombre cuyo dtype inferido difiera entre familias.


In [17]:

dtype_rows = []
for fam in FAMILIES:
    for col, dtype in schemas_latest[fam]["processed"]["dtypes"].items():
        dtype_rows.append({"familia": fam, "column": col, "dtype": dtype})
dtype_df = pd.DataFrame(dtype_rows)

pivot_dtype = dtype_df.pivot(index="column", columns="familia", values="dtype")
divergent = pivot_dtype[
    pivot_dtype.apply(lambda r: r.dropna().nunique() > 1, axis=1)
]
display(divergent)


familia,hipotecarios,personales,prendarios,tarjetas
column,,,,
territorioValidez,float64,float64,str,str



## 13. `all_products.csv`

Se verifica que exista, su número de columnas y cómo representa la unión de schemas de las cuatro familias.


In [18]:
from hashlib import sha256

allp = pd.read_csv(LATEST_BASE / "processed" / "all_products.csv")
processed_frames = [
    pd.read_csv(LATEST_BASE / "processed" / f"{fam}.csv")
    for fam in FAMILIES
]
processed_concat = pd.concat(processed_frames, ignore_index=True)

print("rows:", len(allp), "columns:", allp.shape[1])
print("Columnas all_products:")
print(allp.columns.tolist())
print("\nFilas individuales:", {
    fam: len(frame) for fam, frame in zip(FAMILIES, processed_frames)
})
print("Suma:", len(processed_concat))
print("all_products:", len(allp))
print("Coincide cantidad de filas:", len(processed_concat) == len(allp))

columns_match = list(processed_concat.columns) == list(allp.columns)
column_sets_match = set(processed_concat.columns) == set(allp.columns)
print("Coincide orden de columnas:", columns_match)
print("Coincide conjunto de columnas:", column_sets_match)

missing_in_all_products = sorted(set(processed_concat.columns) - set(allp.columns))
extra_in_all_products = sorted(set(allp.columns) - set(processed_concat.columns))
print("Columnas faltantes en all_products:", missing_in_all_products)
print("Columnas adicionales en all_products:", extra_in_all_products)

# Canonicalizar sólo para comparar: no modifica ninguno de los CSV originales.
def row_hashes(df, columns):
    canonical = df.reindex(columns=columns).astype("string").fillna("<NA>")
    return pd.util.hash_pandas_object(canonical, index=False).astype("uint64")

common_columns = [c for c in processed_concat.columns if c in allp.columns]
processed_hashes = row_hashes(processed_concat, common_columns)
all_products_hashes = row_hashes(allp, common_columns)

ordered_hash_equal = (
    len(processed_hashes) == len(all_products_hashes)
    and processed_hashes.equals(all_products_hashes)
)
multiset_hash_equal = (
    len(processed_hashes) == len(all_products_hashes)
    and processed_hashes.sort_values(ignore_index=True).equals(
        all_products_hashes.sort_values(ignore_index=True)
    )
)

processed_content_hash = sha256(
    processed_hashes.to_numpy().tobytes()
).hexdigest()
all_products_content_hash = sha256(
    all_products_hashes.to_numpy().tobytes()
).hexdigest()
processed_multiset_hash = sha256(
    processed_hashes.sort_values(ignore_index=True).to_numpy().tobytes()
).hexdigest()
all_products_multiset_hash = sha256(
    all_products_hashes.sort_values(ignore_index=True).to_numpy().tobytes()
).hexdigest()

print("\nCoincide contenido en el mismo orden:", ordered_hash_equal)
print("Coincide contenido como multiconjunto:", multiset_hash_equal)
print("Hash ordenado processed concatenado:", processed_content_hash)
print("Hash ordenado all_products:", all_products_content_hash)
print("Hash multiconjunto processed concatenado:", processed_multiset_hash)
print("Hash multiconjunto all_products:", all_products_multiset_hash)


rows: 3593 columns: 49
Columnas all_products:
['snapshot_date', 'snapshot_id', 'snapshot_run_at', 'producto_snapshot', 'relacionMontoTasacion', 'destinoFondos', 'denominacion', 'montoMaximoOtorgable', 'plazoMaximoOtorgable', 'ingresoMinimoMensual', 'antiguedadLaboralMinimaMeses', 'edadMaximaSolicitada', 'relacionCuotaIngreso', 'beneficiario', 'cargoMaximoCancelacionAnticipada', 'tasaEfectivaAnualMaxima', 'tipoTasa', 'costoFinancieroEfectivoTotalMaximo', 'cuotaInicial', 'codigoEntidad', 'descripcionEntidad', 'fechaInformacion', 'nombreCompleto', 'nombreCorto', 'territorioValidez', 'masInformacion', 'relacionCuotaIngreso_raw', 'relacionCuotaIngreso_pct', 'relacionMontoTasacion_raw', 'relacionMontoTasacion_pct', 'information_age_days', 'information_age_bucket', 'product_family_key', 'offer_variant_key', 'product_key_version', 'tasaEfectivaAnualMaxima__extreme_gt_1000', 'tasaEfectivaAnualMaxima__negative_or_sentinel', 'costoFinancieroEfectivoTotalMaximo__extreme_gt_1000', 'costoFinancieroE


## 14. Artefactos globales de control

Se inspeccionan:

- `latest_snapshot.json`
- `quality_flags.csv`
- `key_collisions.csv`
- `ratio_normalization_audit.csv`

No se usan todavía para limpiar ni descartar registros.


In [19]:
latest_pointer_live = json.loads(
    (DATA_B / "latest_snapshot.json").read_text(encoding="utf-8")
)

quality_flags_live = pd.read_csv(DATA_B / "quality_flags.csv")
key_collisions_live = pd.read_csv(DATA_B / "key_collisions.csv")
ratio_audit_live = pd.read_csv(DATA_B / "ratio_normalization_audit.csv")

quality_flags = restrict_to_eda_universe(quality_flags_live)
key_collisions = restrict_to_eda_universe(key_collisions_live)
ratio_audit = restrict_to_eda_universe(ratio_audit_live)

latest_included = {
    "snapshot_date": str(LATEST_DATE),
    "snapshot_id": str(LATEST_SID),
    "path": str(LATEST_BASE),
}

print("Último snapshot operativo (latest_snapshot.json):")
display(latest_pointer_live)

print("\nÚltimo snapshot incluido en este EDA:")
display(latest_included)

print("\nArtefactos globales — live vs universo EDA")
artifact_scope = pd.DataFrame([
    {"artifact": "quality_flags", "rows_live": len(quality_flags_live), "rows_eda": len(quality_flags)},
    {"artifact": "key_collisions", "rows_live": len(key_collisions_live), "rows_eda": len(key_collisions)},
    {"artifact": "ratio_normalization_audit", "rows_live": len(ratio_audit_live), "rows_eda": len(ratio_audit)},
])
display(artifact_scope)

print("\nquality_flags:", quality_flags.shape)
display(quality_flags.head(20))

print("\nkey_collisions:", key_collisions.shape)
display(key_collisions.head(20))

print("\nratio_normalization_audit:", ratio_audit.shape)
display(ratio_audit.head(20))


Último snapshot operativo (latest_snapshot.json):


{'snapshot_date': '2026-09-18',
 'snapshot_id': '20260918_225718_879151',
 'run_at': '2026-09-18T22:57:18.879151-03:00',
 'path': 'C:\\Users\\Asus\\OneDrive\\BRIAN\\CAECE\\4°año_2°Cuatri\\PROYECTO FINAL EN CIENCIA DE DATOS\\scripts\\data_b\\snapshots\\2026-09-18\\20260918_225718_879151'}


Último snapshot incluido en este EDA:


{'snapshot_date': '2026-09-18',
 'snapshot_id': '20260918_225718_879151',
 'path': '..\\data_b\\snapshots\\2026-09-18\\20260918_225718_879151'}


Artefactos globales — live vs universo EDA


,artifact,rows_live,rows_eda
0,quality_flags,168,155
1,key_collisions,3898,3598
2,ratio_normalization_audit,65,60


... (2 outputs left out)


key_collisions: (3598, 7)


,snapshot_date,snapshot_id,run_at,producto,key_type,key,rows_with_same_key
0,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_09cc31aeda4e15cf4a3f8d26,2
1,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_0bd2963787713430d57958b5,3
2,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_0d165ab377af36bc39073d17,3
3,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_1cbda670e738879853608143,2
4,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_217f28284d7516cfe443fff4,3
5,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_336cf8a10a6308a5551ddd1f,2
6,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_44d49b3c3da366aa10bd4838,2
7,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_5556e0c6f752dbc5a8abf1b3,2
8,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_581b1487b4983aeba94e020f,2
9,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,product_family_key,fam_a0b3a2ff175a9755145e17fd,2



ratio_normalization_audit: (60, 9)


,snapshot_date,snapshot_id,run_at,producto,column,n_non_null,n_fractional_0_1_converted,n_values_1_100,n_outside_expected_ratio_range
0,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,relacionCuotaIngreso,269,1,268,0
1,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,relacionMontoTasacion,269,1,268,0
2,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,personales,relacionCuotaIngreso,2278,34,2076,0
3,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,relacionCuotaIngreso,490,0,468,0
4,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,relacionMontoTasacion,490,0,471,0
5,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,relacionCuotaIngreso,267,1,266,0
6,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,relacionMontoTasacion,267,1,266,0
7,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,personales,relacionCuotaIngreso,2278,34,2076,0
8,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,prendarios,relacionCuotaIngreso,490,0,468,0
9,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,prendarios,relacionMontoTasacion,490,0,471,0


## 15. Identidad de producto: evidencia inicial

La identidad longitudinal es central. El pipeline ya genera dos niveles:

- `product_family_key`: familia lógica;
- `offer_variant_key`: variante concreta.

El archivo `key_collisions.csv` permite estudiar la multiplicidad de cada clave dentro de un snapshot y a lo largo del tiempo. No todas las repeticiones tienen el mismo significado.

### `product_family_key`: multiplicidad esperable

`product_family_key` no debe tratarse como una clave primaria de fila. Su función es agrupar múltiples variantes de un mismo producto lógico. Por eso, que una familia aparezca en varias filas puede ser correcto y esperable; por ejemplo, Banco Nación puede presentar una misma familia hipotecaria con tres variantes.

La métrica correcta distingue entre ocurrencias registradas y claves distintas. En los 10 snapshots se observan:

| Familia | Ocurrencias de `product_family_key` | Claves distintas |
|---|---:|---:|
| Hipotecarios | 170 | 17 |
| Personales | 2028 | 203 |
| Prendarios | 650 | 65 |
| Tarjetas | 60 | 6 |

Estas repeticiones se interpretan como **family key multiplicity**, no como un error de identidad.

### `offer_variant_key`: no es una PK única

En cambio, `offer_variant_key` sí pretende identificar una variante concreta, pero presenta multiplicidad dentro del mismo snapshot: hay 9 claves distintas afectadas en total, distribuidas así:

- Personales: 6;
- Prendarios: 1;
- Tarjetas: 2;
- Hipotecarios: 0.

Los casos observados muestran que la misma clave puede reunir ofertas con diferencias sustantivas:

- Personales: TEA `87.27` frente a `88.34`;
- Prendarios, `MSM Leasing`: TEA `84` frente a `98`;
- Tarjetas, `CREDIMAZ`: TEA de financiación `149.61` frente a `106.49`;
- Tarjetas, `YPF En Ruta`: diferencias en `CANONFISICO` y `CANONAPP` dentro de `masInformacion`.

Por lo tanto, **`offer_variant_key` no puede utilizarse como PK única de una oferta dentro de un snapshot**. Antes de analizar altas, bajas o modificaciones, el EDA debe resolver cómo distinguir variantes que comparten esa clave y qué atributos estables deben completar la identidad.


In [20]:
key_multiplicity_summary = (
    key_collisions
    .groupby(["producto", "key_type"])
    .agg(
        key_occurrences=("key", "size"),
        distinct_keys=("key", "nunique"),
        snapshots_affected=("snapshot_id", "nunique"),
        max_rows_same_key=("rows_with_same_key", "max"),
        total_rows_in_multiplicity=("rows_with_same_key", "sum")
    )
    .reset_index()
)
display(key_multiplicity_summary)


,producto,key_type,key_occurrences,distinct_keys,snapshots_affected,max_rows_same_key,total_rows_in_multiplicity
0,hipotecarios,product_family_key,204,17,12,3,480
1,personales,offer_variant_key,72,6,12,2,144
2,personales,product_family_key,2434,203,12,6,5056
3,prendarios,offer_variant_key,12,1,12,2,24
4,prendarios,product_family_key,780,65,12,5,1680
5,tarjetas,offer_variant_key,24,2,12,2,48
6,tarjetas,product_family_key,72,6,12,3,156


In [21]:
# Un ejemplo real de product_family_key con múltiples variantes en el último snapshot.
for fam in FAMILIES:
    df = pd.read_csv(LATEST_BASE / "processed" / f"{fam}.csv")
    fam_counts = df["product_family_key"].value_counts()
    repeated = fam_counts[fam_counts > 1]
    print("\n", "="*100)
    print(fam.upper(), "- familias lógicas con >1 fila:", len(repeated))
    if len(repeated):
        key = repeated.index[0]
        cols = [c for c in [
            "codigoEntidad","descripcionEntidad","nombreCompleto","nombreCorto",
            "beneficiario","segmento","destinoFondos","denominacion",
            "product_family_key","offer_variant_key"
        ] if c in df.columns]
        display(df[df.product_family_key == key][cols].head(20))



HIPOTECARIOS - familias lógicas con >1 fila: 17


,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,beneficiario,destinoFondos,denominacion,product_family_key,offer_variant_key
1,11,BANCO DE LA NACION ARGENTINA,PRESTAMOS HIPOTECARIOS EN UVA,SECTOR PUBLICO,Clientes que acrediten sueldos en la entidad,Construcción,UVA,fam_217f28284d7516cfe443fff4,var_3ae1e157de2547657bb6bd13
12,11,BANCO DE LA NACION ARGENTINA,PRESTAMOS HIPOTECARIOS EN UVA,SECTOR PUBLICO,Clientes que acrediten sueldos en la entidad,Construcción,UVA,fam_217f28284d7516cfe443fff4,var_ac68134abb9b1f34ec596a3b
13,11,BANCO DE LA NACION ARGENTINA,PRESTAMOS HIPOTECARIOS EN UVA,SECTOR PUBLICO,Clientes que acrediten sueldos en la entidad,Construcción,UVA,fam_217f28284d7516cfe443fff4,var_dc86565f25ffaf2e99dfec4b



PERSONALES - familias lógicas con >1 fila: 203


,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,beneficiario,denominacion,product_family_key,offer_variant_key
1834,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_da7dc3ab14e129ebd786f144
1838,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_4e7b1a8a91935d1a1ac95a54
1840,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_f28463a7c2af2d3462eacd79
1842,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_eb8c59ea76bc0ad0bf76b0ef
1844,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_2aeff9b7480518f2b39bfebe
1845,55444,CREDISE S.A.,PRESTAMOS SINDICATO DE SEGURIDAD E INVESTIGACIONES PRIVADAS ...,SSIP,Empleados públicos,Pesos,fam_794139fa81ba5ec8c9e23ece,var_640c4b511b5034aee4ed4032



PRENDARIOS - familias lógicas con >1 fila: 65


,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,beneficiario,destinoFondos,denominacion,product_family_key,offer_variant_key
0,7,BANCO DE GALICIA Y BUENOS AIRES S.A.,PRESTAMOPRENDARIO,PRENDARIO,Clientes con cuenta en la entidad,Adquisición de automotores usados,Pesos,fam_e321c455a824e2614a16afbb,var_f89de7174407a449a475a6e7
1,7,BANCO DE GALICIA Y BUENOS AIRES S.A.,PRESTAMOPRENDARIO,PRENDARIO,Clientes con cuenta en la entidad,Adquisición de automotores usados,Pesos,fam_e321c455a824e2614a16afbb,var_e5f3ba3a2ee504660feb5fd3
2,7,BANCO DE GALICIA Y BUENOS AIRES S.A.,PRESTAMOPRENDARIO,PRENDARIO,Clientes con cuenta en la entidad,Adquisición de automotores usados,Pesos,fam_e321c455a824e2614a16afbb,var_e44d9ea54751a00e868c084c
7,7,BANCO DE GALICIA Y BUENOS AIRES S.A.,PRESTAMOPRENDARIO,PRENDARIO,Clientes con cuenta en la entidad,Adquisición de automotores usados,Pesos,fam_e321c455a824e2614a16afbb,var_76f009e59e76defe48795312
17,7,BANCO DE GALICIA Y BUENOS AIRES S.A.,PRESTAMOPRENDARIO,PRENDARIO,Clientes con cuenta en la entidad,Adquisición de automotores usados,Pesos,fam_e321c455a824e2614a16afbb,var_be3cbeaf686c0a1a5de7f1b0



TARJETAS - familias lógicas con >1 fila: 6


,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,segmento,product_family_key,offer_variant_key
527,72603,CREDIMAZ S.R.L.,NACIONAL CREDIMAZ,NACIONAL,Nacional,fam_59b446a68f37593545710cac,var_cad79d0285ee9b08233b77b9
528,72603,CREDIMAZ S.R.L.,NACIONAL CREDIMAZ,NACIONAL,Nacional,fam_59b446a68f37593545710cac,var_5fba82cd4b06e0b29f3f5bda
529,72603,CREDIMAZ S.R.L.,NACIONAL CREDIMAZ,NACIONAL,Nacional,fam_59b446a68f37593545710cac,var_5fba82cd4b06e0b29f3f5bda



# 17. Quality checks de Base B

A partir de este punto el notebook agrega un **Quality Gate técnico/semántico** sobre los mismos snapshots incluidos en el EDA.

Principios:

- no modifica `raw`, `flattened` ni `processed`;
- no elimina ni imputa registros;
- separa reglas duras de diagnósticos heurísticos;
- los valores extremos se marcan para revisión, no se declaran automáticamente erróneos;
- la identidad longitudinal definitiva, altas, bajas y modificaciones quedan para el EDA 02.

Estados utilizados:

- `PASS`: el control no encontró incidencias;
- `REVIEW`: existe un hallazgo que requiere interpretación;
- `FAIL`: existe una inconsistencia objetiva.

Severidades:

- `INFO`;
- `WARNING`;
- `ERROR`.

`BLOCKED` queda reservado para fallas estructurales que impiden utilizar Base B con seguridad. Los hallazgos semánticos no concluyentes producen `PASS_WITH_WARNINGS`.


In [22]:

# Configuración de calidad.
# Los umbrales REVIEW son heurísticos y NO son reglas de limpieza.

QUALITY_THRESHOLDS = {
    "row_incomplete_review_pct": 80.0,
    "stale_information_review_days": 365,
    "rate_extreme_review": 1000.0,
    "ratio_extreme_review_pct": 100.0,
    "max_age_review": 100.0,
    "max_age_semantic_hard": 150.0,
    "max_employment_months_review": 600.0,
    "max_term_review": 600.0,
}

TECH_REQUIRED = [
    "snapshot_id",
    "codigoEntidad",
    "descripcionEntidad",
    "fechaInformacion",
    "product_family_key",
    "offer_variant_key",
]

RATE_COLUMNS = [
    "tasaEfectivaAnualMaxima",
    "costoFinancieroEfectivoTotalMaximo",
    "tasaEfectivaAnualMaximaFinanciacion",
    "tasaEfectivaAnualMaximaAdelantoEfectivo",
]

MONETARY_COLUMNS = [
    "montoMinimoOtorgable",
    "montoMaximoOtorgable",
    "ingresoMinimoMensual",
    "cuotaInicial",
    "cargoMaximoCancelacionAnticipada",
    "comisionMaximaAdministracionMantenimiento",
    "comisionMaximaRenovacion",
]

REQUIREMENT_COLUMNS = [
    "plazoMaximoOtorgable",
    "edadMaximaSolicitada",
    "antiguedadLaboralMinimaMeses",
]

RATIO_PCT_COLUMNS = [
    "relacionCuotaIngreso_pct",
    "relacionMontoTasacion_pct",
]

CATEGORICAL_COLUMNS = [
    "beneficiario",
    "tipoTasa",
    "denominacion",
    "destinoFondos",
    "segmento",
    "territorioValidez",
]

NAME_COLUMNS = [
    "descripcionEntidad",
    "nombreCompleto",
    "nombreCorto",
]

quality_checks = []

def add_check(
    check_id,
    category,
    family="ALL",
    snapshot_id="ALL",
    severity="INFO",
    status="PASS",
    metric=None,
    value=None,
    expected=None,
    details=None,
    blocking=False,
):
    quality_checks.append({
        "check_id": check_id,
        "category": category,
        "family": family,
        "snapshot_id": snapshot_id,
        "severity": severity,
        "status": status,
        "metric": metric,
        "value": value,
        "expected": expected,
        "details": details,
        "blocking": bool(blocking),
    })

def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    value = re.sub(r"\s+", " ", value)
    return value if value else pd.NA

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

# Cargar todos los processed del universo EDA una sola vez.
processed_store = {}
processed_frames_all = []

for _, snap in complete.iterrows():
    d = str(snap["snapshot_date_dir"])
    sid = str(snap["snapshot_id"])
    base = SNAPSHOT_ROOT / d / sid

    for fam in FAMILIES:
        df = pd.read_csv(base / "processed" / f"{fam}.csv")
        processed_store[(sid, fam)] = df

        tmp = df.copy()
        tmp["_eda_family"] = fam
        tmp["_eda_snapshot_id"] = sid
        tmp["_eda_snapshot_date"] = d
        processed_frames_all.append(tmp)

processed_all = pd.concat(processed_frames_all, ignore_index=True, sort=False)

print("Filas processed acumuladas en universo EDA:", len(processed_all))
print("Snapshots completos:", complete["snapshot_id"].nunique())
print("Familias:", FAMILIES)


Filas processed acumuladas en universo EDA: 42994
Snapshots completos: 12
Familias: ['hipotecarios', 'personales', 'prendarios', 'tarjetas']



## 17.A Integridad de registros y completitud

Se revisan:

- duplicados exactos residuales en `processed`;
- campos técnicos/identificatorios faltantes;
- porcentaje de atributos de negocio nulos por fila;
- registros extremadamente incompletos.

El umbral de incompletitud es **heurístico** y sólo genera `REVIEW`.


In [23]:

integrity_rows = []
incomplete_examples = []

for (sid, fam), df in processed_store.items():
    # 1) Duplicados exactos residuales en processed
    residual_dups = int(df.duplicated().sum())
    add_check(
        "processed_exact_duplicates",
        "integrity",
        family=fam,
        snapshot_id=sid,
        severity="ERROR" if residual_dups else "INFO",
        status="FAIL" if residual_dups else "PASS",
        metric="exact_duplicate_rows",
        value=residual_dups,
        expected=0,
        details="Duplicados exactos residuales después del procesamiento.",
        blocking=residual_dups > 0,
    )

    # 2) Campos técnicos/identificatorios
    missing_tech = {}
    for col in TECH_REQUIRED:
        if col not in df.columns:
            missing_tech[col] = len(df)
        else:
            s = df[col]
            blank = s.astype("string").str.strip().eq("").fillna(False)
            missing_tech[col] = int((s.isna() | blank).sum())

    tech_missing_total = int(sum(missing_tech.values()))
    add_check(
        "required_technical_fields",
        "integrity",
        family=fam,
        snapshot_id=sid,
        severity="ERROR" if tech_missing_total else "INFO",
        status="FAIL" if tech_missing_total else "PASS",
        metric="missing_required_values",
        value=tech_missing_total,
        expected=0,
        details=str({k: v for k, v in missing_tech.items() if v}),
        blocking=tech_missing_total > 0,
    )

    # 3) Completitud de atributos BCRA/raw.
    business_cols = [
        c for c in df.columns
        if origin_of(c) == "BCRA/raw"
        and c not in {"codigoEntidad", "descripcionEntidad", "fechaInformacion"}
    ]

    if business_cols:
        business_null_pct = df[business_cols].isna().mean(axis=1) * 100
        threshold = QUALITY_THRESHOLDS["row_incomplete_review_pct"]
        n_high_incomplete = int((business_null_pct >= threshold).sum())

        integrity_rows.append({
            "snapshot_id": sid,
            "familia": fam,
            "rows": len(df),
            "business_columns": len(business_cols),
            "median_business_null_pct": float(business_null_pct.median()),
            "p90_business_null_pct": float(business_null_pct.quantile(0.90)),
            "p99_business_null_pct": float(business_null_pct.quantile(0.99)),
            "max_business_null_pct": float(business_null_pct.max()),
            f"rows_ge_{int(threshold)}pct_null": n_high_incomplete,
        })

        if n_high_incomplete:
            ex = df.loc[business_null_pct >= threshold].copy()
            ex["_business_null_pct"] = business_null_pct[business_null_pct >= threshold]
            ex["_eda_family"] = fam
            ex["_eda_snapshot_id"] = sid
            incomplete_examples.append(ex)

        add_check(
            "row_business_completeness",
            "completeness",
            family=fam,
            snapshot_id=sid,
            severity="WARNING" if n_high_incomplete else "INFO",
            status="REVIEW" if n_high_incomplete else "PASS",
            metric=f"rows_ge_{threshold:.0f}pct_business_null",
            value=n_high_incomplete,
            expected=0,
            details="Umbral heurístico; no implica eliminación.",
            blocking=False,
        )

integrity_summary = pd.DataFrame(integrity_rows)
display(integrity_summary)

if incomplete_examples:
    incomplete_df = pd.concat(incomplete_examples, ignore_index=True, sort=False)
    cols = [c for c in [
        "_eda_snapshot_id", "_eda_family", "_business_null_pct",
        "codigoEntidad", "descripcionEntidad", "nombreCompleto", "nombreCorto",
        "fechaInformacion", "product_family_key", "offer_variant_key"
    ] if c in incomplete_df.columns]
    print("\nEjemplos de registros con alta incompletitud de atributos de negocio:")
    display(
        incomplete_df[cols]
        .sort_values("_business_null_pct", ascending=False)
        .head(30)
    )
else:
    incomplete_df = pd.DataFrame()
    print("No se detectaron registros por encima del umbral heurístico de incompletitud.")


# Resumen sin inflar por repetición temporal.
if len(incomplete_df):
    latest_sid_quality = str(LATEST_SID)
    incomplete_summary_by_family = (
        incomplete_df.groupby("_eda_family")
        .agg(
            accumulated_observations=("_eda_snapshot_id", "size"),
            snapshots_affected=("_eda_snapshot_id", "nunique"),
            distinct_offer_variant_keys=("offer_variant_key", "nunique"),
        )
        .reset_index()
        .rename(columns={"_eda_family": "familia"})
    )
    latest_incomplete = (
        incomplete_df[incomplete_df["_eda_snapshot_id"].astype(str) == latest_sid_quality]
        .groupby("_eda_family")
        .size()
        .rename("rows_latest_snapshot")
        .reset_index()
        .rename(columns={"_eda_family": "familia"})
    )
    incomplete_summary_by_family = incomplete_summary_by_family.merge(
        latest_incomplete, on="familia", how="left"
    )
    incomplete_summary_by_family["rows_latest_snapshot"] = (
        incomplete_summary_by_family["rows_latest_snapshot"].fillna(0).astype(int)
    )
else:
    incomplete_summary_by_family = pd.DataFrame(columns=[
        "familia", "accumulated_observations", "snapshots_affected",
        "distinct_offer_variant_keys", "rows_latest_snapshot"
    ])

print("\\nResumen de incompletitud: observaciones acumuladas vs. casos distintos vs. último snapshot")
display(incomplete_summary_by_family)


,snapshot_id,familia,rows,business_columns,median_business_null_pct,p90_business_null_pct,p99_business_null_pct,max_business_null_pct,rows_ge_80pct_null
0,20260907_212935_569641,hipotecarios,270,19,0.000000,5.263158,5.263158,100.0,1
1,20260907_212935_569641,personales,2282,18,5.555556,5.555556,5.555556,100.0,2
2,20260907_212935_569641,prendarios,491,20,0.000000,5.000000,5.000000,100.0,1
3,20260907_212935_569641,tarjetas,541,12,8.333333,8.333333,16.666667,100.0,2
4,20260908_224655_838862,hipotecarios,268,19,0.000000,5.263158,5.263158,100.0,1
5,20260908_224655_838862,personales,2282,18,5.555556,5.555556,5.555556,100.0,2
6,20260908_224655_838862,prendarios,491,20,0.000000,5.000000,5.000000,100.0,1
7,20260908_224655_838862,tarjetas,541,12,8.333333,8.333333,16.666667,100.0,2
8,20260909_230424_341254,hipotecarios,268,19,0.000000,5.263158,5.263158,100.0,1
9,20260909_230424_341254,personales,2285,18,5.555556,5.555556,5.555556,100.0,2



Ejemplos de registros con alta incompletitud de atributos de negocio:


,_eda_snapshot_id,_eda_family,_business_null_pct,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,fechaInformacion,product_family_key,offer_variant_key
0,20260907_212935_569641,hipotecarios,100.0,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,fam_dd88a59da49bae4e4e34c588,var_a63592ec4f92c7775f8e27ed
2,20260907_212935_569641,personales,100.0,55303,FINANCE GROUP SRL,NaN,NaN,2021-11-30,fam_7cf593508f586e9fb5ec0a8a,var_87c273e4026d4a170edcfbcb
3,20260907_212935_569641,prendarios,100.0,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,fam_0912f34f0311d735d7f3eb6c,var_6829e50df892d39702d9dc0d
5,20260907_212935_569641,tarjetas,100.0,72115,San Cristóbal Caja Mutual entre Asociados de San Cristobal SMSG,NaN,NaN,2020-01-08,fam_8dd1c5d77d371c7de8719267,var_8338260f8301ace914194c46
6,20260908_224655_838862,hipotecarios,100.0,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,fam_dd88a59da49bae4e4e34c588,var_a63592ec4f92c7775f8e27ed
11,20260908_224655_838862,tarjetas,100.0,72115,San Cristóbal Caja Mutual entre Asociados de San Cristobal SMSG,NaN,NaN,2020-01-08,fam_8dd1c5d77d371c7de8719267,var_8338260f8301ace914194c46
9,20260908_224655_838862,prendarios,100.0,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,fam_0912f34f0311d735d7f3eb6c,var_6829e50df892d39702d9dc0d
8,20260908_224655_838862,personales,100.0,55303,FINANCE GROUP SRL,NaN,NaN,2021-11-30,fam_7cf593508f586e9fb5ec0a8a,var_87c273e4026d4a170edcfbcb
71,20260918_225718_879151,tarjetas,100.0,72115,San Cristóbal Caja Mutual entre Asociados de San Cristobal SMSG,NaN,NaN,2020-01-08,fam_8dd1c5d77d371c7de8719267,var_8338260f8301ace914194c46
68,20260918_225718_879151,personales,100.0,55303,FINANCE GROUP SRL,NaN,NaN,2021-11-30,fam_7cf593508f586e9fb5ec0a8a,var_87c273e4026d4a170edcfbcb


\nResumen de incompletitud: observaciones acumuladas vs. casos distintos vs. último snapshot


,familia,accumulated_observations,snapshots_affected,distinct_offer_variant_keys,rows_latest_snapshot
0,hipotecarios,12,12,1,1
1,personales,24,12,2,2
2,prendarios,12,12,1,1
3,tarjetas,24,12,2,2



## 17.B Consistencia de entidades

Se controla la relación:

- `codigoEntidad → descripcionEntidad`;
- nombre normalizado de entidad → códigos observados.

La normalización se utiliza **sólo para diagnóstico** (`strip`, mayúsculas y colapso de espacios). No modifica la base.


In [24]:

entity_pairs = (
    processed_all[
        ["_eda_family", "codigoEntidad", "descripcionEntidad"]
    ]
    .dropna(subset=["codigoEntidad", "descripcionEntidad"])
    .drop_duplicates()
    .copy()
)
entity_pairs["descripcionEntidad_norm"] = entity_pairs["descripcionEntidad"].map(normalize_text)

code_to_names = (
    entity_pairs.groupby(["_eda_family", "codigoEntidad"])
    .agg(
        raw_names=("descripcionEntidad", lambda s: sorted(set(map(str, s)))),
        n_raw_names=("descripcionEntidad", "nunique"),
        norm_names=("descripcionEntidad_norm", lambda s: sorted(set(s.dropna().astype(str)))),
        n_norm_names=("descripcionEntidad_norm", "nunique"),
    )
    .reset_index()
)
code_name_conflicts = code_to_names[code_to_names["n_norm_names"] > 1].copy()

name_to_codes = (
    entity_pairs.dropna(subset=["descripcionEntidad_norm"])
    .groupby(["_eda_family", "descripcionEntidad_norm"])
    .agg(
        codes=("codigoEntidad", lambda s: sorted(set(map(str, s)))),
        n_codes=("codigoEntidad", "nunique"),
        raw_names=("descripcionEntidad", lambda s: sorted(set(map(str, s)))),
    )
    .reset_index()
)
name_code_conflicts = name_to_codes[name_to_codes["n_codes"] > 1].copy()

for fam in FAMILIES:
    n_code_conflicts = int((code_name_conflicts["_eda_family"] == fam).sum())
    n_name_conflicts = int((name_code_conflicts["_eda_family"] == fam).sum())

    add_check(
        "entity_code_to_name",
        "entities",
        family=fam,
        severity="WARNING" if n_code_conflicts else "INFO",
        status="REVIEW" if n_code_conflicts else "PASS",
        metric="codes_with_multiple_normalized_names",
        value=n_code_conflicts,
        expected=0,
        details="Puede reflejar cambios de denominación; revisar, no corregir automáticamente.",
    )
    add_check(
        "entity_name_to_code",
        "entities",
        family=fam,
        severity="WARNING" if n_name_conflicts else "INFO",
        status="REVIEW" if n_name_conflicts else "PASS",
        metric="normalized_names_with_multiple_codes",
        value=n_name_conflicts,
        expected=0,
        details="Puede reflejar homónimos o cambios de código; requiere revisión.",
    )

print("Código de entidad con más de un nombre normalizado:")
display(code_name_conflicts)

print("\nNombre normalizado asociado a más de un código:")
display(name_code_conflicts)


Código de entidad con más de un nombre normalizado:


,_eda_family,codigoEntidad,raw_names,n_raw_names,norm_names,n_norm_names



Nombre normalizado asociado a más de un código:


,_eda_family,descripcionEntidad_norm,codes,n_codes,raw_names



## 17.C Calidad temporal

Controles:

- parseo de `fechaInformacion`;
- `fechaInformacion <= snapshot_date`;
- consistencia de `information_age_days`;
- distribución de antigüedad;
- información muy antigua.

El umbral de antigüedad es heurístico y genera `REVIEW`, no `FAIL`.


In [25]:

date_quality_rows = []
stale_examples = []

for (sid, fam), df in processed_store.items():
    snap_date_series = pd.to_datetime(df["snapshot_date"], errors="coerce") if "snapshot_date" in df.columns else pd.Series(pd.NaT, index=df.index)
    info_date_series = pd.to_datetime(df["fechaInformacion"], errors="coerce") if "fechaInformacion" in df.columns else pd.Series(pd.NaT, index=df.index)

    invalid_snapshot_dates = int(snap_date_series.isna().sum())
    invalid_info_dates = int(info_date_series.isna().sum())

    valid_both = snap_date_series.notna() & info_date_series.notna()
    computed_age = (snap_date_series - info_date_series).dt.days
    future_mask = valid_both & (computed_age < 0)
    future_count = int(future_mask.sum())

    age_mismatch = 0
    if "information_age_days" in df.columns:
        reported_age = safe_numeric(df["information_age_days"])
        compare = valid_both & reported_age.notna()
        age_mismatch = int((reported_age[compare] != computed_age[compare]).sum())

    stale_threshold = QUALITY_THRESHOLDS["stale_information_review_days"]
    stale_mask = valid_both & (computed_age > stale_threshold)
    stale_count = int(stale_mask.sum())

    valid_age = computed_age[valid_both]
    date_quality_rows.append({
        "snapshot_id": sid,
        "familia": fam,
        "rows": len(df),
        "invalid_snapshot_date": invalid_snapshot_dates,
        "invalid_fechaInformacion": invalid_info_dates,
        "future_fechaInformacion": future_count,
        "information_age_mismatch": age_mismatch,
        "age_min": float(valid_age.min()) if len(valid_age) else np.nan,
        "age_median": float(valid_age.median()) if len(valid_age) else np.nan,
        "age_p90": float(valid_age.quantile(0.90)) if len(valid_age) else np.nan,
        "age_p99": float(valid_age.quantile(0.99)) if len(valid_age) else np.nan,
        "age_max": float(valid_age.max()) if len(valid_age) else np.nan,
        f"age_gt_{stale_threshold}d": stale_count,
    })

    invalid_total = invalid_snapshot_dates + invalid_info_dates
    add_check(
        "date_parseability",
        "dates",
        family=fam,
        snapshot_id=sid,
        severity="ERROR" if invalid_total else "INFO",
        status="FAIL" if invalid_total else "PASS",
        metric="invalid_date_values",
        value=invalid_total,
        expected=0,
        blocking=invalid_total > 0,
    )
    add_check(
        "future_information_date",
        "dates",
        family=fam,
        snapshot_id=sid,
        severity="ERROR" if future_count else "INFO",
        status="FAIL" if future_count else "PASS",
        metric="future_fechaInformacion",
        value=future_count,
        expected=0,
        details="fechaInformacion posterior al snapshot.",
        blocking=False,
    )
    add_check(
        "information_age_consistency",
        "dates",
        family=fam,
        snapshot_id=sid,
        severity="ERROR" if age_mismatch else "INFO",
        status="FAIL" if age_mismatch else "PASS",
        metric="information_age_days_mismatch",
        value=age_mismatch,
        expected=0,
        blocking=False,
    )
    add_check(
        "stale_information",
        "dates",
        family=fam,
        snapshot_id=sid,
        severity="WARNING" if stale_count else "INFO",
        status="REVIEW" if stale_count else "PASS",
        metric=f"rows_age_gt_{stale_threshold}d",
        value=stale_count,
        expected="diagnóstico",
        details="Umbral heurístico. Información antigua no equivale automáticamente a oferta inválida.",
    )

    if stale_count:
        ex = df.loc[stale_mask].copy()
        ex["_computed_information_age_days"] = computed_age[stale_mask]
        ex["_eda_family"] = fam
        ex["_eda_snapshot_id"] = sid
        stale_examples.append(ex)

date_quality = pd.DataFrame(date_quality_rows)
display(date_quality)

if stale_examples:
    stale_df = pd.concat(stale_examples, ignore_index=True, sort=False)
    cols = [c for c in [
        "_eda_snapshot_id", "_eda_family", "codigoEntidad", "descripcionEntidad",
        "nombreCompleto", "nombreCorto", "fechaInformacion",
        "_computed_information_age_days", "product_family_key", "offer_variant_key"
    ] if c in stale_df.columns]
    print("\nRegistros con información más antigua (ejemplos):")
    display(
        stale_df[cols]
        .sort_values("_computed_information_age_days", ascending=False)
        .drop_duplicates(subset=[c for c in ["_eda_family", "codigoEntidad", "nombreCompleto", "nombreCorto", "fechaInformacion"] if c in cols])
        .head(30)
    )
else:
    stale_df = pd.DataFrame()


# Resumen de antigüedad evitando confundir observaciones repetidas con productos distintos.
if len(stale_df):
    latest_sid_quality = str(LATEST_SID)
    stale_summary_by_family = (
        stale_df.groupby("_eda_family")
        .agg(
            accumulated_observations=("_eda_snapshot_id", "size"),
            snapshots_affected=("_eda_snapshot_id", "nunique"),
            distinct_offer_variant_keys=("offer_variant_key", "nunique"),
        )
        .reset_index()
        .rename(columns={"_eda_family": "familia"})
    )
    latest_stale = (
        stale_df[stale_df["_eda_snapshot_id"].astype(str) == latest_sid_quality]
        .groupby("_eda_family")
        .size()
        .rename("rows_latest_snapshot")
        .reset_index()
        .rename(columns={"_eda_family": "familia"})
    )
    stale_summary_by_family = stale_summary_by_family.merge(
        latest_stale, on="familia", how="left"
    )
    stale_summary_by_family["rows_latest_snapshot"] = (
        stale_summary_by_family["rows_latest_snapshot"].fillna(0).astype(int)
    )
else:
    stale_summary_by_family = pd.DataFrame(columns=[
        "familia", "accumulated_observations", "snapshots_affected",
        "distinct_offer_variant_keys", "rows_latest_snapshot"
    ])

print("\\nResumen de antigüedad: observaciones acumuladas vs. claves distintas vs. último snapshot")
display(stale_summary_by_family)


,snapshot_id,familia,rows,invalid_snapshot_date,invalid_fechaInformacion,future_fechaInformacion,information_age_mismatch,age_min,age_median,age_p90,age_p99,age_max,age_gt_365d
0,20260907_212935_569641,hipotecarios,270,0,0,0,0,0.0,34.0,123.0,276.34,3441.0,3
1,20260907_212935_569641,personales,2282,0,0,0,0,0.0,75.0,1195.0,1672.34,2317.0,355
2,20260907_212935_569641,prendarios,491,0,0,0,0,0.0,27.0,342.0,1517.00,3441.0,41
3,20260907_212935_569641,tarjetas,541,0,0,0,0,0.0,27.0,368.0,1813.00,2568.0,69
4,20260908_224655_838862,hipotecarios,268,0,0,0,0,0.0,35.0,124.0,281.62,3442.0,3
5,20260908_224655_838862,personales,2282,0,0,0,0,0.0,76.0,1196.0,1673.34,2318.0,355
6,20260908_224655_838862,prendarios,491,0,0,0,0,0.0,28.0,343.0,1518.00,3442.0,41
7,20260908_224655_838862,tarjetas,541,0,0,0,0,0.0,26.0,369.0,1814.00,2569.0,69
8,20260909_230424_341254,hipotecarios,268,0,0,0,0,1.0,36.0,125.0,282.62,3443.0,3
9,20260909_230424_341254,personales,2285,0,0,0,0,0.0,77.0,1197.0,1671.76,2319.0,355



Registros con información más antigua (ejemplos):


,_eda_snapshot_id,_eda_family,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,fechaInformacion,_computed_information_age_days,product_family_key,offer_variant_key
5413,20260918_225718_879151,prendarios,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,3452,fam_0912f34f0311d735d7f3eb6c,var_6829e50df892d39702d9dc0d
5041,20260918_225718_879151,hipotecarios,70427,TARJETA GRUPAR S.A.,NaN,NaN,2017-04-06,3452,fam_dd88a59da49bae4e4e34c588,var_a63592ec4f92c7775f8e27ed
5473,20260918_225718_879151,tarjetas,72122,MUTUAL DE RESIDENTES DEL BARRIO TAIS,EXPRES CARD PLUS,EXPRES CARD PLUS,2019-08-27,2579,fam_848dd2c79a950b39f8332f11,var_7f3f0ec51a8e7be9f25f5ef3
5464,20260918_225718_879151,tarjetas,70427,TARJETA GRUPAR S.A.,TARJETA GRUPAR S.A.,TG,2019-11-20,2494,fam_d7f0869b03d45dc0e576ae16,var_c139321c2453c0f81e0254d4
5470,20260918_225718_879151,tarjetas,72115,San Cristóbal Caja Mutual entre Asociados de San Cristobal SMSG,NaN,NaN,2020-01-08,2445,fam_8dd1c5d77d371c7de8719267,var_8338260f8301ace914194c46
5363,20260918_225718_879151,personales,70427,TARJETA GRUPAR S.A.,PRESTAMO PERSONAL TARJETA GRUPAR,PRESTAMOS GRUPAR 2019,2020-05-04,2328,fam_3e1cbc7ddb35923276d6f0e3,var_72972d85ad75fbbc15c034cc
5459,20260918_225718_879151,tarjetas,70161,FIDEN S.A.,TARJETA COMPUMUNDO MASTERCARD INTERNACIONAL,TARJETA MASTERCARD,2021-04-29,1968,fam_b8ca600aec93d019dda2db1e,var_aa5b72069dc7dd7013499795
5460,20260918_225718_879151,tarjetas,70161,FIDEN S.A.,TARJETA GARBARINO MASTERCARD INTERNACIONAL,TARJETA MASTERCARD,2021-04-29,1968,fam_095c575e88f4bb6b6eb4017b,var_75b486d660332cc653394840
5274,20260918_225718_879151,personales,55277,ASOCIACION MUTUAL FIAT CONCORD,MUNIC CARLOS PAZ JUBIL (2205),MUNIC CARLOS PAZ JUBIL,2021-05-20,1947,fam_ad486f385354838f93b6993f,var_d1f7352cd23c76dd2fee7f2a
5266,20260918_225718_879151,personales,55277,ASOCIACION MUTUAL FIAT CONCORD,BANCOR SUC C PAZ (11968),BANCOR SUC C PAZ,2021-05-20,1947,fam_7b4bf4731c391c870a90b5aa,var_aaa9c2f3b2672542dc30a54d


\nResumen de antigüedad: observaciones acumuladas vs. claves distintas vs. último snapshot


,familia,accumulated_observations,snapshots_affected,distinct_offer_variant_keys,rows_latest_snapshot
0,hipotecarios,34,12,3,1
1,personales,4129,12,358,331
2,prendarios,492,12,40,41
3,tarjetas,828,12,68,69



## 17.D Valores numéricos, ceros y relaciones lógicas

Reglas duras:

- valores negativos en tasas, montos y requisitos que conceptualmente no deberían ser negativos;
- `montoMinimoOtorgable > montoMaximoOtorgable`.

Diagnósticos:

- valores `0`;
- tasas extremadamente altas;
- edad, antigüedad laboral y plazo muy altos.

Los umbrales extremos son heurísticos.


In [26]:

numeric_quality_rows = []
zero_rows = []
distribution_rows = []
logical_issues = []
zero_max_review_issues = []
semantic_outlier_examples = []

candidate_numeric_columns = RATE_COLUMNS + MONETARY_COLUMNS + REQUIREMENT_COLUMNS + RATIO_PCT_COLUMNS

for fam in FAMILIES:
    fam_df = processed_all[processed_all["_eda_family"] == fam].copy()

    for col in candidate_numeric_columns:
        if col not in fam_df.columns:
            continue

        x = safe_numeric(fam_df[col])
        valid = x.dropna()
        if valid.empty:
            continue

        n_negative = int((valid < 0).sum())
        n_zero = int((valid == 0).sum())

        distribution_rows.append({
            "familia": fam,
            "columna": col,
            "n": int(valid.size),
            "min": float(valid.min()),
            "p01": float(valid.quantile(0.01)),
            "p25": float(valid.quantile(0.25)),
            "median": float(valid.median()),
            "p75": float(valid.quantile(0.75)),
            "p99": float(valid.quantile(0.99)),
            "max": float(valid.max()),
        })

        zero_rows.append({
            "familia": fam,
            "columna": col,
            "non_null": int(valid.size),
            "zeros": n_zero,
            "zero_pct_non_null": float(n_zero / valid.size * 100),
        })

        # Negativos: objetivamente requieren tratamiento para modelado.
        add_check(
            f"negative_{col}",
            "numeric_values",
            family=fam,
            severity="ERROR" if n_negative else "INFO",
            status="FAIL" if n_negative else "PASS",
            metric="negative_values",
            value=n_negative,
            expected=0,
            details="Puede incluir sentinels de fuente; no se elimina automáticamente.",
            blocking=False,
        )

        # Extremos heurísticos: REVIEW, no error.
        # Para edad, los valores por encima del máximo semántico duro se separan más abajo.
        threshold = None
        hard_upper = None
        if col in RATE_COLUMNS:
            threshold = QUALITY_THRESHOLDS["rate_extreme_review"]
        elif col == "edadMaximaSolicitada":
            threshold = QUALITY_THRESHOLDS["max_age_review"]
            hard_upper = QUALITY_THRESHOLDS["max_age_semantic_hard"]
        elif col == "antiguedadLaboralMinimaMeses":
            threshold = QUALITY_THRESHOLDS["max_employment_months_review"]
        elif col == "plazoMaximoOtorgable":
            threshold = QUALITY_THRESHOLDS["max_term_review"]
        elif col in RATIO_PCT_COLUMNS:
            threshold = QUALITY_THRESHOLDS["ratio_extreme_review_pct"]

        if threshold is not None:
            if hard_upper is not None:
                review_mask = (valid > threshold) & (valid <= hard_upper)
            else:
                review_mask = valid > threshold
            n_extreme = int(review_mask.sum())
            add_check(
                f"extreme_{col}",
                "numeric_values",
                family=fam,
                severity="WARNING" if n_extreme else "INFO",
                status="REVIEW" if n_extreme else "PASS",
                metric=(
                    f"values_gt_{threshold}_le_{hard_upper}"
                    if hard_upper is not None
                    else f"values_gt_{threshold}"
                ),
                value=n_extreme,
                expected="diagnóstico",
                details="Umbral heurístico; revisar plausibilidad y unidad.",
            )

        # Outlier semántico duro: edad humana claramente imposible.
        if col == "edadMaximaSolicitada":
            semantic_limit = QUALITY_THRESHOLDS["max_age_semantic_hard"]
            semantic_mask = x.notna() & (x > semantic_limit)
            n_semantic = int(semantic_mask.sum())
            add_check(
                "semantic_outlier_age",
                "semantic_outlier",
                family=fam,
                severity="ERROR" if n_semantic else "INFO",
                status="FAIL" if n_semantic else "PASS",
                metric=f"edadMaximaSolicitada_gt_{semantic_limit}",
                value=n_semantic,
                expected=0,
                details=(
                    "Valor incompatible con una edad humana plausible. "
                    "Se preserva en la fuente y se excluye/transforma sólo en una capa posterior."
                ),
                blocking=False,
            )
            if n_semantic:
                ex = fam_df.loc[semantic_mask, [
                    c for c in [
                        "_eda_snapshot_id", "codigoEntidad", "descripcionEntidad",
                        "nombreCompleto", "nombreCorto", "edadMaximaSolicitada",
                        "offer_variant_key"
                    ] if c in fam_df.columns
                ]].copy()
                ex["familia"] = fam
                ex["semantic_issue"] = f"edadMaximaSolicitada > {semantic_limit}"
                semantic_outlier_examples.append(ex)

    # Relación monto mínimo <= máximo.
    # FIX: montoMaximo==0 no se interpreta automáticamente como máximo literal.
    if {"montoMinimoOtorgable", "montoMaximoOtorgable"}.issubset(fam_df.columns):
        mn = safe_numeric(fam_df["montoMinimoOtorgable"])
        mx = safe_numeric(fam_df["montoMaximoOtorgable"])

        comparable = mn.notna() & mx.notna()
        true_inconsistency = comparable & (mx > 0) & (mn > mx)
        zero_max_semantics = comparable & (mn > 0) & (mx == 0)

        n_bad = int(true_inconsistency.sum())
        n_zero_max = int(zero_max_semantics.sum())

        add_check(
            "min_amount_le_max_amount_positive_max",
            "numeric_values",
            family=fam,
            severity="ERROR" if n_bad else "INFO",
            status="FAIL" if n_bad else "PASS",
            metric="rows_min_gt_positive_max",
            value=n_bad,
            expected=0,
            details="Sólo se evalúa cuando montoMaximoOtorgable > 0.",
            blocking=False,
        )

        add_check(
            "zero_max_amount_semantics",
            "numeric_values",
            family=fam,
            severity="WARNING" if n_zero_max else "INFO",
            status="REVIEW" if n_zero_max else "PASS",
            metric="rows_positive_min_with_zero_max",
            value=n_zero_max,
            expected="semántica de 0 por validar",
            details=(
                "montoMaximoOtorgable=0 no se interpreta como un máximo literal. "
                "Puede significar sin máximo/no informado/sentinel según la fuente."
            ),
            blocking=False,
        )

        if n_bad:
            ex = fam_df.loc[true_inconsistency, [
                c for c in [
                    "_eda_snapshot_id", "codigoEntidad", "descripcionEntidad",
                    "nombreCompleto", "nombreCorto",
                    "montoMinimoOtorgable", "montoMaximoOtorgable",
                    "offer_variant_key"
                ] if c in fam_df.columns
            ]].copy()
            ex["familia"] = fam
            ex["issue_type"] = "min_gt_positive_max"
            logical_issues.append(ex)

        if n_zero_max:
            ex = fam_df.loc[zero_max_semantics, [
                c for c in [
                    "_eda_snapshot_id", "codigoEntidad", "descripcionEntidad",
                    "nombreCompleto", "nombreCorto",
                    "montoMinimoOtorgable", "montoMaximoOtorgable",
                    "offer_variant_key"
                ] if c in fam_df.columns
            ]].copy()
            ex["familia"] = fam
            ex["issue_type"] = "positive_min_zero_max_review"
            zero_max_review_issues.append(ex)

numeric_distribution = pd.DataFrame(distribution_rows)
zero_summary = pd.DataFrame(zero_rows)

print("Distribuciones numéricas:")
display(numeric_distribution)

print("\nDiagnóstico de ceros (no se clasifican automáticamente como error):")
display(zero_summary.sort_values(["familia", "zero_pct_non_null"], ascending=[True, False]))

if logical_issues:
    logical_issues_df = pd.concat(logical_issues, ignore_index=True, sort=False)
    print("\nInconsistencias objetivas: monto mínimo > monto máximo positivo")
    display(logical_issues_df.head(100))
else:
    logical_issues_df = pd.DataFrame()
    print("\nNo se detectaron casos monto mínimo > monto máximo positivo.")

if zero_max_review_issues:
    zero_max_review_df = pd.concat(zero_max_review_issues, ignore_index=True, sort=False)
    print("\nCasos para REVIEW: monto mínimo positivo con monto máximo = 0")
    display(zero_max_review_df.head(100))
else:
    zero_max_review_df = pd.DataFrame()

if semantic_outlier_examples:
    semantic_outlier_df = pd.concat(semantic_outlier_examples, ignore_index=True, sort=False)
    print("\nOutliers semánticos duros:")
    display(semantic_outlier_df.head(100))
else:
    semantic_outlier_df = pd.DataFrame()
    print("\nNo se detectaron outliers semánticos duros según las reglas actuales.")


# Resumen de reglas de monto y outliers semánticos sin inflar por repetición temporal.
def summarize_issue_scope(df, issue_name):
    if df is None or df.empty:
        return {
            "issue": issue_name,
            "accumulated_observations": 0,
            "distinct_offer_variant_keys": 0,
            "rows_latest_snapshot": 0,
        }
    return {
        "issue": issue_name,
        "accumulated_observations": int(len(df)),
        "distinct_offer_variant_keys": int(df["offer_variant_key"].nunique()) if "offer_variant_key" in df.columns else 0,
        "rows_latest_snapshot": int((df["_eda_snapshot_id"].astype(str) == str(LATEST_SID)).sum()) if "_eda_snapshot_id" in df.columns else 0,
    }

numeric_issue_scope_summary = pd.DataFrame([
    summarize_issue_scope(logical_issues_df, "min_gt_positive_max"),
    summarize_issue_scope(zero_max_review_df, "positive_min_zero_max_review"),
    summarize_issue_scope(semantic_outlier_df, "semantic_outlier_age"),
])

print("\\nResumen de incidencias numéricas: acumuladas vs. claves distintas vs. último snapshot")
display(numeric_issue_scope_summary)


Distribuciones numéricas:


,familia,columna,n,min,p01,p25,median,p75,p99,max
0,hipotecarios,tasaEfectivaAnualMaxima,3211,3.56,4.28,9.920000e+00,1.300000e+01,2.819000e+01,4.595000e+01,6.959000e+01
1,hipotecarios,costoFinancieroEfectivoTotalMaximo,3211,3.56,4.28,1.039000e+01,1.500000e+01,2.828000e+01,5.902000e+01,6.959000e+01
2,hipotecarios,montoMaximoOtorgable,3211,200000.00,30000000.00,2.500000e+08,3.257116e+08,3.721133e+08,1.000000e+10,1.000000e+10
3,hipotecarios,ingresoMinimoMensual,3211,0.00,0.00,5.000000e+04,7.532000e+05,1.506400e+06,9.900000e+06,1.000000e+07
4,hipotecarios,cuotaInicial,3211,10.00,82.40,2.025000e+03,6.802000e+03,1.321000e+04,8.331300e+04,6.196120e+05
5,hipotecarios,cargoMaximoCancelacionAnticipada,3211,0.00,0.00,1.000000e+00,3.000000e+00,4.000000e+00,4.790000e+00,4.790000e+00
6,hipotecarios,plazoMaximoOtorgable,3211,48.00,60.00,1.800000e+02,2.400000e+02,2.400000e+02,3.600000e+02,3.600000e+02
7,hipotecarios,edadMaximaSolicitada,3211,45.00,45.00,5.900000e+01,6.500000e+01,8.500000e+01,1.000000e+02,1.000000e+02
8,hipotecarios,antiguedadLaboralMinimaMeses,3211,0.00,0.00,1.200000e+01,1.200000e+01,1.800000e+01,3.600000e+01,3.600000e+01
9,hipotecarios,relacionCuotaIngreso_pct,3211,20.00,25.00,2.500000e+01,2.500000e+01,3.000000e+01,5.000000e+01,5.000000e+01



Diagnóstico de ceros (no se clasifican automáticamente como error):


,familia,columna,non_null,zeros,zero_pct_non_null
3,hipotecarios,ingresoMinimoMensual,3211,768,23.917783
5,hipotecarios,cargoMaximoCancelacionAnticipada,3211,312,9.716599
8,hipotecarios,antiguedadLaboralMinimaMeses,3211,36,1.121146
0,hipotecarios,tasaEfectivaAnualMaxima,3211,0,0.000000
1,hipotecarios,costoFinancieroEfectivoTotalMaximo,3211,0,0.000000
2,hipotecarios,montoMaximoOtorgable,3211,0,0.000000
4,hipotecarios,cuotaInicial,3211,0,0.000000
6,hipotecarios,plazoMaximoOtorgable,3211,0,0.000000
7,hipotecarios,edadMaximaSolicitada,3211,0,0.000000
9,hipotecarios,relacionCuotaIngreso_pct,3211,0,0.000000



Inconsistencias objetivas: monto mínimo > monto máximo positivo


,_eda_snapshot_id,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,montoMinimoOtorgable,montoMaximoOtorgable,offer_variant_key,familia,issue_type
0,20260907_212935_569641,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,N 0KM PRES PERS 12 PV JUL26,N 0KM PRES PERS 12 PV JUL,600000.0,500000.0,var_69ba70b4bfa74e382e5ee0b2,personales,min_gt_positive_max
1,20260907_212935_569641,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,N USADO PRES PERS 12PV JUL26,N USADO PRES PERS 12PV JU,600000.0,500000.0,var_8b079847e3f99ae84eda44d7,personales,min_gt_positive_max
2,20260908_224655_838862,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,N 0KM PRES PERS 12 PV JUL26,N 0KM PRES PERS 12 PV JUL,600000.0,500000.0,var_69ba70b4bfa74e382e5ee0b2,personales,min_gt_positive_max
3,20260908_224655_838862,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,N USADO PRES PERS 12PV JUL26,N USADO PRES PERS 12PV JU,600000.0,500000.0,var_8b079847e3f99ae84eda44d7,personales,min_gt_positive_max
4,20260909_230424_341254,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max
5,20260910_221306_387342,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max
6,20260911_215454_602912,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max
7,20260912_192218_082470,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max
8,20260913_220415_571963,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max
9,20260914_231738_476390,44095,ROMBO COMPAÑÍA FINANCIERA S.A.,R 0KM CARO EMPLEADO 24M AGO26,R 0KM CARO EMPLEADO 24M A,600000.0,500000.0,var_8482cf8c8d6a200bd745284a,personales,min_gt_positive_max


... (2 outputs left out)


Outliers semánticos duros:


,_eda_snapshot_id,codigoEntidad,descripcionEntidad,nombreCompleto,nombreCorto,edadMaximaSolicitada,offer_variant_key,familia,semantic_issue
0,20260907_212935_569641,70270,Asociación Mutual Club Social Deportivo Ascensión,ASOCIACION MUTUAL CLUB SOCIAL DEPORTIVO ASCENSION,TARJETA GAUCHA,1000.0,var_4d6f0d82ea4608c546054c00,tarjetas,edadMaximaSolicitada > 150.0
1,20260907_212935_569641,72128,ASOCIACION MUTUAL CLUB COLON DE SAN JUSTO,TARJETA MUTUAL MAS,TARJETA,8000.0,var_5f6dc1820abad80444801bbe,tarjetas,edadMaximaSolicitada > 150.0
2,20260908_224655_838862,70270,Asociación Mutual Club Social Deportivo Ascensión,ASOCIACION MUTUAL CLUB SOCIAL DEPORTIVO ASCENSION,TARJETA GAUCHA,1000.0,var_4d6f0d82ea4608c546054c00,tarjetas,edadMaximaSolicitada > 150.0
3,20260908_224655_838862,72128,ASOCIACION MUTUAL CLUB COLON DE SAN JUSTO,TARJETA MUTUAL MAS,TARJETA,8000.0,var_5f6dc1820abad80444801bbe,tarjetas,edadMaximaSolicitada > 150.0
4,20260909_230424_341254,70270,Asociación Mutual Club Social Deportivo Ascensión,ASOCIACION MUTUAL CLUB SOCIAL DEPORTIVO ASCENSION,TARJETA GAUCHA,1000.0,var_4d6f0d82ea4608c546054c00,tarjetas,edadMaximaSolicitada > 150.0
5,20260909_230424_341254,72128,ASOCIACION MUTUAL CLUB COLON DE SAN JUSTO,TARJETA MUTUAL MAS,TARJETA,8000.0,var_5f6dc1820abad80444801bbe,tarjetas,edadMaximaSolicitada > 150.0
6,20260910_221306_387342,70270,Asociación Mutual Club Social Deportivo Ascensión,ASOCIACION MUTUAL CLUB SOCIAL DEPORTIVO ASCENSION,TARJETA GAUCHA,1000.0,var_4d6f0d82ea4608c546054c00,tarjetas,edadMaximaSolicitada > 150.0
7,20260910_221306_387342,72128,ASOCIACION MUTUAL CLUB COLON DE SAN JUSTO,TARJETA MUTUAL MAS,TARJETA,8000.0,var_5f6dc1820abad80444801bbe,tarjetas,edadMaximaSolicitada > 150.0
8,20260911_215454_602912,70270,Asociación Mutual Club Social Deportivo Ascensión,ASOCIACION MUTUAL CLUB SOCIAL DEPORTIVO ASCENSION,TARJETA GAUCHA,1000.0,var_4d6f0d82ea4608c546054c00,tarjetas,edadMaximaSolicitada > 150.0
9,20260911_215454_602912,72128,ASOCIACION MUTUAL CLUB COLON DE SAN JUSTO,TARJETA MUTUAL MAS,TARJETA,8000.0,var_5f6dc1820abad80444801bbe,tarjetas,edadMaximaSolicitada > 150.0


\nResumen de incidencias numéricas: acumuladas vs. claves distintas vs. último snapshot


,issue,accumulated_observations,distinct_offer_variant_keys,rows_latest_snapshot
0,min_gt_positive_max,14,3,1
1,positive_min_zero_max_review,996,139,83
2,semantic_outlier_age,24,2,2



## 17.E Tasas y CFTEA

Además de negativos/extremos, se revisa:

- `CFTEA < TEA` cuando ambas variables existen;
- flags técnicos generados por el pipeline (`__extreme_gt_1000`, `__negative_or_sentinel`).

`CFTEA < TEA` se marca como `REVIEW`, no como error automático.


In [27]:

rate_relation_rows = []
pipeline_flag_rows = []

for fam in FAMILIES:
    fam_df = processed_all[processed_all["_eda_family"] == fam].copy()

    # Relaciones CFTEA vs TEA en préstamos.
    if {"tasaEfectivaAnualMaxima", "costoFinancieroEfectivoTotalMaximo"}.issubset(fam_df.columns):
        tea = safe_numeric(fam_df["tasaEfectivaAnualMaxima"])
        cftea = safe_numeric(fam_df["costoFinancieroEfectivoTotalMaximo"])
        comparable = tea.notna() & cftea.notna()
        mask = comparable & (cftea < tea)
        n_bad = int(mask.sum())

        rate_relation_rows.append({
            "familia": fam,
            "comparables": int(comparable.sum()),
            "cftea_lt_tea": n_bad,
            "pct": float(n_bad / comparable.sum() * 100) if comparable.sum() else np.nan,
        })

        add_check(
            "cftea_ge_tea",
            "rates",
            family=fam,
            severity="WARNING" if n_bad else "INFO",
            status="REVIEW" if n_bad else "PASS",
            metric="rows_cftea_lt_tea",
            value=n_bad,
            expected="revisar semántica",
            details="No se considera FAIL hasta validar semántica exacta del endpoint.",
        )

    # Flags ya producidos por el extractor.
    flag_cols = [
        c for c in fam_df.columns
        if c.endswith("__extreme_gt_1000")
        or c.endswith("__negative_or_sentinel")
    ]
    for col in flag_cols:
        s = fam_df[col]
        if s.dtype == bool:
            n_flagged = int(s.sum())
        else:
            normalized = s.astype("string").str.lower().str.strip()
            n_flagged = int(normalized.isin(["true", "1", "yes"]).sum())

        pipeline_flag_rows.append({
            "familia": fam,
            "flag": col,
            "flagged_rows": n_flagged,
        })

        add_check(
            f"pipeline_flag_{col}",
            "rates",
            family=fam,
            severity="WARNING" if n_flagged else "INFO",
            status="REVIEW" if n_flagged else "PASS",
            metric="flagged_rows",
            value=n_flagged,
            expected=0,
            details="Flag del extractor; requiere interpretación antes de limpiar.",
        )

rate_relations = pd.DataFrame(rate_relation_rows)
pipeline_flags_summary = pd.DataFrame(pipeline_flag_rows)

print("Relación CFTEA / TEA:")
display(rate_relations)

print("\nFlags técnicos del pipeline:")
display(pipeline_flags_summary)


Relación CFTEA / TEA:


,familia,comparables,cftea_lt_tea,pct
0,hipotecarios,3211,12,0.373715
1,personales,27366,336,1.227801
2,prendarios,5885,12,0.203908
3,tarjetas,0,0,NaN



Flags técnicos del pipeline:


,familia,flag,flagged_rows
0,hipotecarios,tasaEfectivaAnualMaxima__extreme_gt_1000,0
1,hipotecarios,tasaEfectivaAnualMaxima__negative_or_sentinel,0
2,hipotecarios,costoFinancieroEfectivoTotalMaximo__extreme_gt_1000,0
3,hipotecarios,costoFinancieroEfectivoTotalMaximo__negative_or_sentinel,0
4,hipotecarios,tasaEfectivaAnualMaximaFinanciacion__extreme_gt_1000,0
5,hipotecarios,tasaEfectivaAnualMaximaFinanciacion__negative_or_sentinel,0
6,hipotecarios,tasaEfectivaAnualMaximaAdelantoEfectivo__extreme_gt_1000,0
7,hipotecarios,tasaEfectivaAnualMaximaAdelantoEfectivo__negative_or_sentinel,0
8,personales,tasaEfectivaAnualMaxima__extreme_gt_1000,1322
9,personales,tasaEfectivaAnualMaxima__negative_or_sentinel,0



## 17.F Ratios y normalización

Se valida:

- consistencia de presencia `raw ↔ pct`;
- porcentajes negativos;
- porcentajes altos;
- auditoría existente en `ratio_normalization_audit.csv`.

No se redefine aquí la regla de normalización del extractor.


In [28]:

ratio_quality_rows = []

ratio_pairs = [
    ("relacionCuotaIngreso_raw", "relacionCuotaIngreso_pct"),
    ("relacionMontoTasacion_raw", "relacionMontoTasacion_pct"),
]

for fam in FAMILIES:
    fam_df = processed_all[processed_all["_eda_family"] == fam].copy()

    for raw_col_name, pct_col_name in ratio_pairs:
        if raw_col_name not in fam_df.columns or pct_col_name not in fam_df.columns:
            continue

        raw = safe_numeric(fam_df[raw_col_name])
        pct = safe_numeric(fam_df[pct_col_name])

        raw_only = int((raw.notna() & pct.isna()).sum())
        pct_only = int((raw.isna() & pct.notna()).sum())
        negative_pct = int((pct.dropna() < 0).sum())
        extreme_threshold = QUALITY_THRESHOLDS["ratio_extreme_review_pct"]
        high_pct = int((pct.dropna() > extreme_threshold).sum())

        ratio_quality_rows.append({
            "familia": fam,
            "raw_col": raw_col_name,
            "pct_col": pct_col_name,
            "raw_present_pct_missing": raw_only,
            "raw_missing_pct_present": pct_only,
            "pct_negative": negative_pct,
            f"pct_gt_{extreme_threshold}": high_pct,
        })

        missing_pair = raw_only + pct_only
        add_check(
            f"ratio_presence_{pct_col_name}",
            "ratios",
            family=fam,
            severity="ERROR" if missing_pair else "INFO",
            status="FAIL" if missing_pair else "PASS",
            metric="raw_pct_presence_mismatch",
            value=missing_pair,
            expected=0,
        )
        add_check(
            f"ratio_negative_{pct_col_name}",
            "ratios",
            family=fam,
            severity="ERROR" if negative_pct else "INFO",
            status="FAIL" if negative_pct else "PASS",
            metric="negative_pct",
            value=negative_pct,
            expected=0,
        )
        add_check(
            f"ratio_high_{pct_col_name}",
            "ratios",
            family=fam,
            severity="WARNING" if high_pct else "INFO",
            status="REVIEW" if high_pct else "PASS",
            metric=f"pct_gt_{extreme_threshold}",
            value=high_pct,
            expected="diagnóstico",
        )

ratio_quality = pd.DataFrame(ratio_quality_rows)
display(ratio_quality)

print("\nAuditoría generada por el extractor:")
print("Columnas:", ratio_audit.columns.tolist())
display(ratio_audit.head(50))

if "action" in ratio_audit.columns:
    print("\nResumen por acción:")
    display(ratio_audit["action"].value_counts(dropna=False).to_frame("rows"))


,familia,raw_col,pct_col,raw_present_pct_missing,raw_missing_pct_present,pct_negative,pct_gt_100.0
0,hipotecarios,relacionCuotaIngreso_raw,relacionCuotaIngreso_pct,0,0,0,0
1,hipotecarios,relacionMontoTasacion_raw,relacionMontoTasacion_pct,0,0,0,0
2,personales,relacionCuotaIngreso_raw,relacionCuotaIngreso_pct,0,0,0,0
3,personales,relacionMontoTasacion_raw,relacionMontoTasacion_pct,0,0,0,0
4,prendarios,relacionCuotaIngreso_raw,relacionCuotaIngreso_pct,0,0,0,0
5,prendarios,relacionMontoTasacion_raw,relacionMontoTasacion_pct,0,0,0,0
6,tarjetas,relacionCuotaIngreso_raw,relacionCuotaIngreso_pct,0,0,0,0
7,tarjetas,relacionMontoTasacion_raw,relacionMontoTasacion_pct,0,0,0,0



Auditoría generada por el extractor:
Columnas: ['snapshot_date', 'snapshot_id', 'run_at', 'producto', 'column', 'n_non_null', 'n_fractional_0_1_converted', 'n_values_1_100', 'n_outside_expected_ratio_range']


,snapshot_date,snapshot_id,run_at,producto,column,n_non_null,n_fractional_0_1_converted,n_values_1_100,n_outside_expected_ratio_range
0,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,relacionCuotaIngreso,269,1,268,0
1,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,hipotecarios,relacionMontoTasacion,269,1,268,0
2,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,personales,relacionCuotaIngreso,2278,34,2076,0
3,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,relacionCuotaIngreso,490,0,468,0
4,2026-09-07,20260907_212935_569641,2026-09-07T21:29:35.569641-03:00,prendarios,relacionMontoTasacion,490,0,471,0
5,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,relacionCuotaIngreso,267,1,266,0
6,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,hipotecarios,relacionMontoTasacion,267,1,266,0
7,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,personales,relacionCuotaIngreso,2278,34,2076,0
8,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,prendarios,relacionCuotaIngreso,490,0,468,0
9,2026-09-08,20260908_224655_838862,2026-09-08T22:46:55.838862-03:00,prendarios,relacionMontoTasacion,490,0,471,0



## 17.G Categorías, nombres y codificaciones

Diagnóstico de:

- cardinalidad;
- strings vacíos;
- variantes que colapsan al normalizar espacios/mayúsculas;
- codificación de `territorioValidez`.

Las variantes se muestran para revisión. No se homologan automáticamente.


In [29]:

category_rows = []
normalization_collisions = []
territory_examples = []

for fam in FAMILIES:
    fam_df = processed_all[processed_all["_eda_family"] == fam].copy()

    for col in CATEGORICAL_COLUMNS + NAME_COLUMNS:
        if col not in fam_df.columns:
            continue

        s = fam_df[col]
        non_null = s.dropna().astype(str)
        blank_count = int(non_null.str.strip().eq("").sum())
        normalized = non_null.map(normalize_text)

        category_rows.append({
            "familia": fam,
            "columna": col,
            "non_null": int(non_null.size),
            "raw_cardinality": int(non_null.nunique()),
            "normalized_cardinality": int(normalized.nunique(dropna=True)),
            "blank_strings": blank_count,
        })

        variant_map = (
            pd.DataFrame({"raw": non_null.values, "norm": normalized.values})
            .dropna(subset=["norm"])
            .drop_duplicates()
            .groupby("norm")
            .agg(
                variants=("raw", lambda x: sorted(set(map(str, x)))),
                n_variants=("raw", "nunique"),
            )
            .reset_index()
        )
        variant_map = variant_map[variant_map["n_variants"] > 1]
        if len(variant_map):
            variant_map["familia"] = fam
            variant_map["columna"] = col
            normalization_collisions.append(variant_map)

        if col == "territorioValidez":
            vals = (
                s.dropna()
                .astype(str)
                .value_counts()
                .head(20)
                .rename_axis("valor")
                .reset_index(name="count")
            )
            vals["familia"] = fam
            territory_examples.append(vals)

        add_check(
            f"blank_strings_{col}",
            "categories",
            family=fam,
            severity="WARNING" if blank_count else "INFO",
            status="REVIEW" if blank_count else "PASS",
            metric="blank_strings",
            value=blank_count,
            expected=0,
        )

category_summary = pd.DataFrame(category_rows)
display(category_summary)

if normalization_collisions:
    normalization_collisions_df = pd.concat(normalization_collisions, ignore_index=True)
    print("\nVariantes que colapsan con normalización diagnóstica:")
    display(normalization_collisions_df.head(100))
else:
    normalization_collisions_df = pd.DataFrame()
    print("\nNo se detectaron variantes puramente de espacios/mayúsculas en los campos revisados.")

if territory_examples:
    territory_values = pd.concat(territory_examples, ignore_index=True)
    print("\nValores frecuentes de territorioValidez por familia:")
    display(territory_values)
else:
    territory_values = pd.DataFrame()


,familia,columna,non_null,raw_cardinality,normalized_cardinality,blank_strings
0,hipotecarios,beneficiario,3211,7,7,0
1,hipotecarios,tipoTasa,3211,3,3,0
2,hipotecarios,denominacion,3211,3,3,0
3,hipotecarios,destinoFondos,3211,5,5,0
4,hipotecarios,segmento,0,0,0,0
5,hipotecarios,territorioValidez,3211,12,12,0
6,hipotecarios,descripcionEntidad,3223,30,30,0
7,hipotecarios,nombreCompleto,3211,65,64,0
8,hipotecarios,nombreCorto,3211,58,58,0
9,personales,beneficiario,27366,8,8,0



Variantes que colapsan con normalización diagnóstica:


,norm,variants,n_variants,familia,columna
0,PH ADQUISICION DE VIVIENDA PERMANENTE EN UVAS,"[PH ADQUISICION DE VIVIENDA PERMANENTE EN UVAS, PH ADQUISICION DE VIVIENDA PERMANENTE EN UVAS ]",2,hipotecarios,nombreCompleto
1,PRESTAMO CONSUMO,"[PRESTAMO CONSUMO, PRESTAMO CONSUMO ...",2,personales,nombreCompleto
2,PRESTAMO EFECTIVO,"[PRESTAMO EFECTIVO, PRESTAMO EFECTIVO ...",2,personales,nombreCompleto
3,CONSUMO,"[CONSUMO, CONSUMO ]",2,personales,nombreCorto
4,EFECTIVO,"[EFECTIVO, EFECTIVO ]",2,personales,nombreCorto
5,MENDOZA,"[MENDOZA, MENDOZA ]",2,personales,nombreCorto
6,PP SIN ACREDITACION HAB,"[PP SIN ACREDITACION HAB, PP SIN ACREDITACION HAB]",2,personales,nombreCorto
7,PREMIUM,"[ PREMIUM, PREMIUM]",2,personales,nombreCorto
8,CREDINET PRENDARIOS 0KM,"[CREDINET PRENDARIOS 0KM, CREDINET PRENDARIOS 0KM ...",2,prendarios,nombreCompleto
9,CREDINET PRENDARIOS USADOS,"[CREDINET PRENDARIOS USADOS, CREDINET PRENDARIOS USADOS ...",2,prendarios,nombreCompleto



Valores frecuentes de territorioValidez por familia:


,valor,count,familia
0,0.0,1553,hipotecarios
1,1.0,757,hipotecarios
2,2.0,480,hipotecarios
3,4.0,144,hipotecarios
4,15.0,144,hipotecarios
5,21.0,72,hipotecarios
6,18.0,12,hipotecarios
7,7.0,12,hipotecarios
8,20.0,12,hipotecarios
9,23.0,12,hipotecarios



## 17.H Identidad intra-snapshot

Este control se calcula directamente sobre `processed`, sin depender únicamente de `key_collisions.csv`.

Se distingue:

- multiplicidad de `product_family_key`: esperable como agrupador lógico;
- multiplicidad de `offer_variant_key`: requiere revisión si se pretende usar como identificador único.

Para las `offer_variant_key` no únicas se muestran los atributos que difieren.


In [30]:

identity_rows = []
offer_collision_examples = []

META_COMPARE_EXCLUDE = {
    "snapshot_date", "snapshot_id", "snapshot_run_at", "producto_snapshot",
    "information_age_days", "information_age_bucket",
    "product_key_version", "product_family_key", "offer_variant_key",
}

for (sid, fam), df in processed_store.items():
    for key_col in ["product_family_key", "offer_variant_key"]:
        if key_col not in df.columns:
            continue

        counts = df[key_col].value_counts(dropna=False)
        repeated = counts[counts > 1]
        n_keys = int(len(repeated))
        rows_in_mult = int(repeated.sum()) if len(repeated) else 0
        max_mult = int(repeated.max()) if len(repeated) else 1

        identity_rows.append({
            "snapshot_id": sid,
            "familia": fam,
            "key_type": key_col,
            "distinct_keys_with_multiplicity": n_keys,
            "rows_in_multiplicity": rows_in_mult,
            "max_multiplicity": max_mult,
        })

        if key_col == "offer_variant_key":
            add_check(
                "offer_variant_key_uniqueness",
                "identity",
                family=fam,
                snapshot_id=sid,
                severity="WARNING" if n_keys else "INFO",
                status="REVIEW" if n_keys else "PASS",
                metric="non_unique_offer_variant_keys",
                value=n_keys,
                expected=0,
                details="Impide tratar offer_variant_key como PK única hasta resolver la identidad.",
            )

            for key in repeated.index:
                if pd.isna(key):
                    continue
                g = df[df[key_col] == key].copy()
                compare_cols = [
                    c for c in g.columns
                    if c not in META_COMPARE_EXCLUDE
                    and not c.endswith("__extreme_gt_1000")
                    and not c.endswith("__negative_or_sentinel")
                ]

                differing = []
                for c in compare_cols:
                    normalized = g[c].astype("string").fillna("<NA>")
                    if normalized.nunique(dropna=False) > 1:
                        differing.append(c)

                offer_collision_examples.append({
                    "snapshot_id": sid,
                    "familia": fam,
                    "offer_variant_key": key,
                    "rows": len(g),
                    "differing_columns": differing,
                })

identity_summary = pd.DataFrame(identity_rows)
display(identity_summary)

offer_collision_differences = pd.DataFrame(offer_collision_examples)

if len(offer_collision_differences):
    print("\nOffer variant keys no únicas y columnas que distinguen sus filas:")
    display(offer_collision_differences.head(100))

    # Vista consolidada por clave a través del tiempo.
    offer_collision_by_key = (
        offer_collision_differences
        .groupby(["familia", "offer_variant_key"])
        .agg(
            snapshots_affected=("snapshot_id", "nunique"),
            max_rows=("rows", "max"),
            differing_columns=(
                "differing_columns",
                lambda s: sorted(set(c for cols in s for c in cols))
            ),
        )
        .reset_index()
    )
    print("\nConsolidado de claves ambiguas:")
    display(offer_collision_by_key)
else:
    offer_collision_by_key = pd.DataFrame()
    print("No se detectaron offer_variant_key no únicas.")


,snapshot_id,familia,key_type,distinct_keys_with_multiplicity,rows_in_multiplicity,max_multiplicity
0,20260907_212935_569641,hipotecarios,product_family_key,17,40,3
1,20260907_212935_569641,hipotecarios,offer_variant_key,0,0,1
2,20260907_212935_569641,personales,product_family_key,202,418,6
3,20260907_212935_569641,personales,offer_variant_key,6,12,2
4,20260907_212935_569641,prendarios,product_family_key,65,140,5
5,20260907_212935_569641,prendarios,offer_variant_key,1,2,2
6,20260907_212935_569641,tarjetas,product_family_key,6,13,3
7,20260907_212935_569641,tarjetas,offer_variant_key,2,4,2
8,20260908_224655_838862,hipotecarios,product_family_key,17,40,3
9,20260908_224655_838862,hipotecarios,offer_variant_key,0,0,1



Offer variant keys no únicas y columnas que distinguen sus filas:


,snapshot_id,familia,offer_variant_key,rows,differing_columns
0,20260907_212935_569641,personales,var_d0a6a8b55cbf35104e532649,2,[masInformacion]
1,20260907_212935_569641,personales,var_185d90cd2c9d2e5ee94f88b0,2,[masInformacion]
2,20260907_212935_569641,personales,var_8059652237ac14727593b5d1,2,[masInformacion]
3,20260907_212935_569641,personales,var_2c90e0b20c62819b6a9be6af,2,[tasaEfectivaAnualMaxima]
4,20260907_212935_569641,personales,var_89b8bfa7f7d23bcc62c7406d,2,[tasaEfectivaAnualMaxima]
5,20260907_212935_569641,personales,var_c53ad74292b2b74a4fe747d2,2,[tasaEfectivaAnualMaxima]
6,20260907_212935_569641,prendarios,var_f976f316aeab5a13427af673,2,[tasaEfectivaAnualMaxima]
7,20260907_212935_569641,tarjetas,var_865d38376e0a1d4f4e20fb80,2,"[comisionMaximaAdministracionMantenimiento, masInformacion]"
8,20260907_212935_569641,tarjetas,var_5fba82cd4b06e0b29f3f5bda,2,"[tasaEfectivaAnualMaximaFinanciacion, tasaEfectivaAnualMaximaAdelantoEfectivo]"
9,20260908_224655_838862,personales,var_d0a6a8b55cbf35104e532649,2,[masInformacion]



Consolidado de claves ambiguas:


,familia,offer_variant_key,snapshots_affected,max_rows,differing_columns
0,personales,var_185d90cd2c9d2e5ee94f88b0,12,2,[masInformacion]
1,personales,var_2c90e0b20c62819b6a9be6af,12,2,[tasaEfectivaAnualMaxima]
2,personales,var_8059652237ac14727593b5d1,12,2,[masInformacion]
3,personales,var_89b8bfa7f7d23bcc62c7406d,12,2,[tasaEfectivaAnualMaxima]
4,personales,var_c53ad74292b2b74a4fe747d2,12,2,[tasaEfectivaAnualMaxima]
5,personales,var_d0a6a8b55cbf35104e532649,12,2,[masInformacion]
6,prendarios,var_f976f316aeab5a13427af673,12,2,[tasaEfectivaAnualMaxima]
7,tarjetas,var_5fba82cd4b06e0b29f3f5bda,12,2,"[tasaEfectivaAnualMaximaAdelantoEfectivo, tasaEfectivaAnualMaximaFinanciacion]"
8,tarjetas,var_865d38376e0a1d4f4e20fb80,12,2,"[comisionMaximaAdministracionMantenimiento, masInformacion]"



## 17.I Quality Gate final

El Quality Gate integra:

- controles estructurales ya realizados en el EDA;
- checks estáticos/semánticos de esta sección.

Estados finales:

- `PASS`: sin hallazgos pendientes;
- `PASS_WITH_WARNINGS`: base estructuralmente utilizable, pero con hallazgos que deben documentarse/tratarse;
- `BLOCKED`: existe una falla estructural bloqueante.

Los `REVIEW` no implican eliminación de registros.


In [31]:

# Incorporar al framework los controles estructurales previos.

# Manifest / HTTP
manifest_expected_rows = int(complete["snapshot_id"].nunique()) * len(FAMILIES)
add_check(
    "manifest_expected_rows",
    "structure",
    severity="ERROR" if len(manifest) != manifest_expected_rows else "INFO",
    status="FAIL" if len(manifest) != manifest_expected_rows else "PASS",
    metric="manifest_rows",
    value=len(manifest),
    expected=manifest_expected_rows,
    blocking=len(manifest) != manifest_expected_rows,
)

bad_http_count = int((manifest[status_col] != 200).sum()) if status_col else 0
add_check(
    "http_status",
    "structure",
    severity="ERROR" if bad_http_count else "INFO",
    status="FAIL" if bad_http_count else "PASS",
    metric="non_200_rows",
    value=bad_http_count,
    expected=0,
    blocking=bad_http_count > 0,
)

# Schema raw / flattened / processed
raw_schema_ok_q = bool(
    (raw_schema_summary["distinct_result_schemas"] == 1).all()
    and (raw_schema_summary["distinct_top_level_schemas"] == 1).all()
)
processed_schema_ok_q = bool(
    (schema_stability["distinct_headers"] == 1).all()
    and (schema_stability["distinct_dtype_signatures"] == 1).all()
)
add_check(
    "raw_schema_stability",
    "structure",
    severity="ERROR" if not raw_schema_ok_q else "INFO",
    status="FAIL" if not raw_schema_ok_q else "PASS",
    metric="stable",
    value=raw_schema_ok_q,
    expected=True,
    blocking=not raw_schema_ok_q,
)
add_check(
    "processed_schema_stability",
    "structure",
    severity="ERROR" if not processed_schema_ok_q else "INFO",
    status="FAIL" if not processed_schema_ok_q else "PASS",
    metric="stable",
    value=processed_schema_ok_q,
    expected=True,
    blocking=not processed_schema_ok_q,
)

# Integridad all_products del último snapshot incluido
all_products_ok_q = bool(
    len(processed_concat) == len(allp)
    and columns_match
    and column_sets_match
    and ordered_hash_equal
    and multiset_hash_equal
)
add_check(
    "all_products_integrity",
    "structure",
    severity="ERROR" if not all_products_ok_q else "INFO",
    status="FAIL" if not all_products_ok_q else "PASS",
    metric="exact_integrity",
    value=all_products_ok_q,
    expected=True,
    blocking=not all_products_ok_q,
)

quality_checks_df = pd.DataFrame(quality_checks)

# Resumen por categoría y estado.
quality_gate_table = (
    quality_checks_df
    .groupby(["category", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["PASS", "REVIEW", "FAIL"], fill_value=0)
    .reset_index()
)
display(quality_gate_table)

blocking_failures = quality_checks_df[
    (quality_checks_df["status"] == "FAIL")
    & quality_checks_df["blocking"]
]
reviews = quality_checks_df[quality_checks_df["status"] == "REVIEW"]
nonblocking_failures = quality_checks_df[
    (quality_checks_df["status"] == "FAIL")
    & ~quality_checks_df["blocking"]
]

if len(blocking_failures):
    QUALITY_GATE_STATUS = "BLOCKED"
elif len(reviews) or len(nonblocking_failures):
    QUALITY_GATE_STATUS = "PASS_WITH_WARNINGS"
else:
    QUALITY_GATE_STATUS = "PASS"

print("\nQUALITY GATE:", QUALITY_GATE_STATUS)
print("Checks totales:", len(quality_checks_df))
print("PASS:", int((quality_checks_df["status"] == "PASS").sum()))
print("REVIEW:", len(reviews))
print("FAIL no bloqueantes:", len(nonblocking_failures))
print("FAIL bloqueantes:", len(blocking_failures))

if len(blocking_failures):
    print("\nFallas bloqueantes:")
    display(blocking_failures)

if len(nonblocking_failures):
    print("\nFallas no bloqueantes que requieren tratamiento/documentación:")
    display(nonblocking_failures.head(100))

if len(reviews):
    print("\nHallazgos para revisión:")
    display(reviews.head(100))


status,category,PASS,REVIEW,FAIL
0,categories,36,0,0
1,completeness,0,48,0
2,dates,144,48,0
3,entities,8,0,0
4,identity,12,36,0
5,integrity,96,0,0
6,numeric_values,67,5,1
7,rates,30,6,0
8,ratios,24,0,0
9,semantic_outlier,3,0,1



QUALITY GATE: PASS_WITH_WARNINGS
Checks totales: 570
PASS: 425
REVIEW: 143
FAIL no bloqueantes: 2
FAIL bloqueantes: 0

Fallas no bloqueantes que requieren tratamiento/documentación:


,check_id,category,family,snapshot_id,severity,status,metric,value,expected,details,blocking
383,min_amount_le_max_amount_positive_max,numeric_values,personales,ALL,ERROR,FAIL,rows_min_gt_positive_max,14,0,Sólo se evalúa cuando montoMaximoOtorgable > 0.,False
416,semantic_outlier_age,semantic_outlier,tarjetas,ALL,ERROR,FAIL,edadMaximaSolicitada_gt_150.0,24,0,Valor incompatible con una edad humana plausible. Se preserva en la fuente y se excluye/transforma sólo en una capa ...,False



Hallazgos para revisión:


,check_id,category,family,snapshot_id,severity,status,metric,value,expected,details,blocking
2,row_business_completeness,completeness,hipotecarios,20260907_212935_569641,WARNING,REVIEW,rows_ge_80pct_business_null,1,0,Umbral heurístico; no implica eliminación.,False
5,row_business_completeness,completeness,personales,20260907_212935_569641,WARNING,REVIEW,rows_ge_80pct_business_null,2,0,Umbral heurístico; no implica eliminación.,False
8,row_business_completeness,completeness,prendarios,20260907_212935_569641,WARNING,REVIEW,rows_ge_80pct_business_null,1,0,Umbral heurístico; no implica eliminación.,False
11,row_business_completeness,completeness,tarjetas,20260907_212935_569641,WARNING,REVIEW,rows_ge_80pct_business_null,2,0,Umbral heurístico; no implica eliminación.,False
14,row_business_completeness,completeness,hipotecarios,20260908_224655_838862,WARNING,REVIEW,rows_ge_80pct_business_null,1,0,Umbral heurístico; no implica eliminación.,False
17,row_business_completeness,completeness,personales,20260908_224655_838862,WARNING,REVIEW,rows_ge_80pct_business_null,2,0,Umbral heurístico; no implica eliminación.,False
20,row_business_completeness,completeness,prendarios,20260908_224655_838862,WARNING,REVIEW,rows_ge_80pct_business_null,1,0,Umbral heurístico; no implica eliminación.,False
23,row_business_completeness,completeness,tarjetas,20260908_224655_838862,WARNING,REVIEW,rows_ge_80pct_business_null,2,0,Umbral heurístico; no implica eliminación.,False
26,row_business_completeness,completeness,hipotecarios,20260909_230424_341254,WARNING,REVIEW,rows_ge_80pct_business_null,1,0,Umbral heurístico; no implica eliminación.,False
29,row_business_completeness,completeness,personales,20260909_230424_341254,WARNING,REVIEW,rows_ge_80pct_business_null,2,0,Umbral heurístico; no implica eliminación.,False



# 18. Conclusiones dinámicas del EDA 01 + Calidad estática

Esta sección se construye con los outputs del notebook. No hardcodea cantidad de snapshots ni fecha final.


In [32]:

# Universo
n_snapshot_dirs = len(physical_inventory)
n_complete = int(physical_inventory["physically_complete"].sum())
n_incomplete = int((~physical_inventory["physically_complete"]).sum())
n_dates = int(coverage["snapshot_date_dir"].nunique())
date_min = coverage["snapshot_date_dir"].min().date().isoformat()
date_max = coverage["snapshot_date_dir"].max().date().isoformat()

multi_days_df = coverage[coverage["n_complete_snapshots"] > 1].copy()
multi_days = [
    f"{r.snapshot_date_dir.date().isoformat()} ({int(r.n_complete_snapshots)} snapshots)"
    for _, r in multi_days_df.iterrows()
]
multi_days_txt = "; ".join(multi_days) if multi_days else "ninguna"
missing_days_txt = ", ".join(map(str, missing_days)) if missing_days else "ninguna"

# Hallazgos específicos
n_offer_ambiguous_keys = (
    int(offer_collision_by_key["offer_variant_key"].nunique())
    if len(offer_collision_by_key)
    else 0
)
offer_ambiguous_by_family = (
    offer_collision_by_key.groupby("familia")["offer_variant_key"].nunique().to_dict()
    if len(offer_collision_by_key)
    else {}
)
offer_lines = "\n".join(
    f"- {fam}: {offer_ambiguous_by_family.get(fam, 0)}"
    for fam in FAMILIES
)

n_code_name_conflicts = len(code_name_conflicts)
n_name_code_conflicts = len(name_code_conflicts)

# Contadores de calidad en tres niveles: observaciones acumuladas, claves distintas, último snapshot.
def quality_scope_totals(summary_df):
    if summary_df is None or summary_df.empty:
        return 0, 0, 0
    return (
        int(summary_df["accumulated_observations"].sum()),
        int(summary_df["distinct_offer_variant_keys"].sum()),
        int(summary_df["rows_latest_snapshot"].sum()),
    )

incomplete_obs, incomplete_keys, incomplete_latest = quality_scope_totals(incomplete_summary_by_family)
stale_obs, stale_keys, stale_latest = quality_scope_totals(stale_summary_by_family)

semantic_obs = len(semantic_outlier_df) if len(semantic_outlier_df) else 0
semantic_keys = (
    int(semantic_outlier_df["offer_variant_key"].nunique())
    if len(semantic_outlier_df) and "offer_variant_key" in semantic_outlier_df.columns
    else 0
)
semantic_latest = (
    int((semantic_outlier_df["_eda_snapshot_id"].astype(str) == str(LATEST_SID)).sum())
    if len(semantic_outlier_df) and "_eda_snapshot_id" in semantic_outlier_df.columns
    else 0
)

def get_issue_scope(issue_name):
    row = numeric_issue_scope_summary[numeric_issue_scope_summary["issue"] == issue_name]
    if row.empty:
        return 0, 0, 0
    r = row.iloc[0]
    return (
        int(r["accumulated_observations"]),
        int(r["distinct_offer_variant_keys"]),
        int(r["rows_latest_snapshot"]),
    )

amount_fail_obs, amount_fail_keys, amount_fail_latest = get_issue_scope("min_gt_positive_max")
amount_zero_review_obs, amount_zero_review_keys, amount_zero_review_latest = get_issue_scope("positive_min_zero_max_review")

blocking_n = len(blocking_failures)
review_n = len(reviews)
nonblocking_fail_n = len(nonblocking_failures)

conclusion_md = f"""
## 18.1 Universo y estructura

- Snapshots incluidos: **{n_snapshot_dirs}**.
- Snapshots completos: **{n_complete}**.
- Corridas parciales/incompletas: **{n_incomplete}**.
- Fechas calendario: **{n_dates}**, desde **{date_min}** hasta **{date_max}**.
- Fechas faltantes dentro del intervalo: **{missing_days_txt}**.
- Fechas con más de un snapshot: **{multi_days_txt}**.
- Schema raw estable: **{raw_schema_ok_q}**.
- Schema flattened/processed estable: **{processed_schema_ok_q}**.
- Integridad exacta de `all_products.csv` en el último snapshot: **{all_products_ok_q}**.

## 18.2 Calidad estática/semántica

El notebook audita de forma reproducible:

- duplicados residuales en `processed`;
- campos técnicos obligatorios observacionales;
- incompletitud por fila;
- consistencia código/nombre de entidad;
- fechas inválidas/futuras y antigüedad de información;
- negativos, ceros y extremos numéricos;
- relación monto mínimo/máximo con semántica especial para máximo `0`;
- outliers semánticos duros separados de outliers económicos;
- tasas y relación CFTEA/TEA;
- ratios raw/pct;
- categorías, strings vacíos y variantes de nombres;
- identidad intra-snapshot.

Casos de nombres/códigos de entidad para revisión:

- códigos con múltiples nombres normalizados: **{n_code_name_conflicts}**;
- nombres normalizados asociados a múltiples códigos: **{n_name_code_conflicts}**.

### Incompletitud

- observaciones acumuladas snapshot-producto por encima del umbral: **{incomplete_obs}**;
- `offer_variant_key` distintas afectadas: **{incomplete_keys}**;
- filas afectadas en el último snapshot: **{incomplete_latest}**.

### Antigüedad de información

- observaciones acumuladas snapshot-producto por encima del umbral: **{stale_obs}**;
- `offer_variant_key` distintas afectadas: **{stale_keys}**;
- filas afectadas en el último snapshot: **{stale_latest}**.

Estos tres niveles evitan interpretar observaciones repetidas entre snapshots como productos distintos.

### Monto mínimo / máximo

Inconsistencias objetivas `montoMinimo > montoMaximo > 0`:

- observaciones acumuladas: **{amount_fail_obs}**;
- `offer_variant_key` distintas: **{amount_fail_keys}**;
- filas afectadas en el último snapshot: **{amount_fail_latest}**.

Casos `montoMinimo > 0` y `montoMaximo = 0` (`REVIEW` hasta resolver la semántica de `0`):

- observaciones acumuladas: **{amount_zero_review_obs}**;
- `offer_variant_key` distintas: **{amount_zero_review_keys}**;
- filas afectadas en el último snapshot: **{amount_zero_review_latest}**.

### Outliers semánticos

- observaciones acumuladas: **{semantic_obs}**;
- `offer_variant_key` distintas: **{semantic_keys}**;
- filas afectadas en el último snapshot: **{semantic_latest}**.

Los outliers económicos plausibles —por ejemplo tasas muy altas— permanecen como `REVIEW`; los valores incompatibles con la semántica de la variable se separan en `semantic_outlier`.

## 18.3 Identidad intra-snapshot

`product_family_key` se interpreta como agrupador lógico, por lo que su multiplicidad no es por sí misma un error.

Se detectan **{n_offer_ambiguous_keys} `offer_variant_key` distintas con multiplicidad intra-snapshot**:

{offer_lines}

Mientras existan estas multiplicidades, `offer_variant_key` **no debe utilizarse como PK única** para calcular altas, bajas y modificaciones.

## 18.4 Quality Gate

**Estado: `{QUALITY_GATE_STATUS}`**

- fallas bloqueantes: **{blocking_n}**;
- fallas no bloqueantes: **{nonblocking_fail_n}**;
- checks en revisión: **{review_n}**.

`PASS_WITH_WARNINGS` significa que la estructura es utilizable, pero existen hallazgos semánticos que deben conservarse como flags o resolverse antes del modelado. No implica que deban eliminarse filas.

## 18.5 Alcance del cierre

Con este notebook queda cubierto:

**inventario → extracción → estructura → schema → completitud → entidades → fechas → numéricos → tasas → ratios → categorías → integridad de artefactos → identidad intra-snapshot → Quality Gate.**

Todavía queda fuera de alcance la **calidad longitudinal**:

- estabilidad de identidad entre snapshots;
- resolución conceptual definitiva de las `offer_variant_key` ambiguas;
- altas y bajas;
- modificaciones de condiciones;
- desapariciones/reapariciones;
- cambios masivos y posibles cambios de fuente.

Esos puntos corresponden al EDA 02.
"""
display(Markdown(conclusion_md))



## 18.1 Universo y estructura

- Snapshots incluidos: **12**.
- Snapshots completos: **12**.
- Corridas parciales/incompletas: **0**.
- Fechas calendario: **12**, desde **2026-09-07** hasta **2026-09-18**.
- Fechas faltantes dentro del intervalo: **ninguna**.
- Fechas con más de un snapshot: **ninguna**.
- Schema raw estable: **True**.
- Schema flattened/processed estable: **True**.
- Integridad exacta de `all_products.csv` en el último snapshot: **True**.

## 18.2 Calidad estática/semántica

El notebook audita de forma reproducible:

- duplicados residuales en `processed`;
- campos técnicos obligatorios observacionales;
- incompletitud por fila;
- consistencia código/nombre de entidad;
- fechas inválidas/futuras y antigüedad de información;
- negativos, ceros y extremos numéricos;
- relación monto mínimo/máximo con semántica especial para máximo `0`;
- outliers semánticos duros separados de outliers económicos;
- tasas y relación CFTEA/TEA;
- ratios raw/pct;
- categorías, strings vacíos y variantes de nombres;
- identidad intra-snapshot.

Casos de nombres/códigos de entidad para revisión:

- códigos con múltiples nombres normalizados: **0**;
- nombres normalizados asociados a múltiples códigos: **0**.

### Incompletitud

- observaciones acumuladas snapshot-producto por encima del umbral: **72**;
- `offer_variant_key` distintas afectadas: **6**;
- filas afectadas en el último snapshot: **6**.

### Antigüedad de información

- observaciones acumuladas snapshot-producto por encima del umbral: **5483**;
- `offer_variant_key` distintas afectadas: **469**;
- filas afectadas en el último snapshot: **442**.

Estos tres niveles evitan interpretar observaciones repetidas entre snapshots como productos distintos.

### Monto mínimo / máximo

Inconsistencias objetivas `montoMinimo > montoMaximo > 0`:

- observaciones acumuladas: **14**;
- `offer_variant_key` distintas: **3**;
- filas afectadas en el último snapshot: **1**.

Casos `montoMinimo > 0` y `montoMaximo = 0` (`REVIEW` hasta resolver la semántica de `0`):

- observaciones acumuladas: **996**;
- `offer_variant_key` distintas: **139**;
- filas afectadas en el último snapshot: **83**.

### Outliers semánticos

- observaciones acumuladas: **24**;
- `offer_variant_key` distintas: **2**;
- filas afectadas en el último snapshot: **2**.

Los outliers económicos plausibles —por ejemplo tasas muy altas— permanecen como `REVIEW`; los valores incompatibles con la semántica de la variable se separan en `semantic_outlier`.

## 18.3 Identidad intra-snapshot

`product_family_key` se interpreta como agrupador lógico, por lo que su multiplicidad no es por sí misma un error.

Se detectan **9 `offer_variant_key` distintas con multiplicidad intra-snapshot**:

- hipotecarios: 0
- personales: 6
- prendarios: 1
- tarjetas: 2

Mientras existan estas multiplicidades, `offer_variant_key` **no debe utilizarse como PK única** para calcular altas, bajas y modificaciones.

## 18.4 Quality Gate

**Estado: `PASS_WITH_WARNINGS`**

- fallas bloqueantes: **0**;
- fallas no bloqueantes: **2**;
- checks en revisión: **143**.

`PASS_WITH_WARNINGS` significa que la estructura es utilizable, pero existen hallazgos semánticos que deben conservarse como flags o resolverse antes del modelado. No implica que deban eliminarse filas.

## 18.5 Alcance del cierre

Con este notebook queda cubierto:

**inventario → extracción → estructura → schema → completitud → entidades → fechas → numéricos → tasas → ratios → categorías → integridad de artefactos → identidad intra-snapshot → Quality Gate.**

Todavía queda fuera de alcance la **calidad longitudinal**:

- estabilidad de identidad entre snapshots;
- resolución conceptual definitiva de las `offer_variant_key` ambiguas;
- altas y bajas;
- modificaciones de condiciones;
- desapariciones/reapariciones;
- cambios masivos y posibles cambios de fuente.

Esos puntos corresponden al EDA 02.



## Próximo notebook

`Base_B_EDA_02_identidad_longitudinal_y_evolucion.ipynb`

Orden recomendado:

1. resolver las `offer_variant_key` ambiguas y determinar atributos faltantes en la identidad;
2. validar estabilidad de `product_family_key` y `offer_variant_key` entre snapshots;
3. construir matching longitudinal;
4. medir productos/ofertas por snapshot;
5. altas y bajas;
6. modificaciones de condiciones;
7. desapariciones y reapariciones;
8. anomalías masivas;
9. separar cambio real de oferta de cambio técnico de fuente.

Sólo después de cerrar identidad longitudinal conviene usar el histórico para modelado de evolución de la oferta.
